# Lab 3: 양자 우위를 위한 새로운 도구

Qiskit Global Summer School 2026의 Lab 3에 오신 것을 환영합니다!

이번 랩에서는 오류 완화(error mitigation) 과정을 한층 정밀하게 제어할 수 있도록 도입된 프레임워크인 __[Samplomatic](https://github.com/Qiskit/samplomatic)__ 사용법을 배웁니다. 양자 우위(Quantum Advantage)를 향해 가는 여정에서 중요한 한 걸음입니다.

- **1장**에서는 Samplomatic *없이* Qiskit Runtime만으로 가능한 오류 완화 방법들을 살펴봅니다. *층을 드레싱한다*(dress a layer)는 것이 무슨 뜻인지, 파울리(Pauli) 에러가 클리포드(Clifford) 게이트를 지나며 어떻게 전파되는지 배웁니다. 2장을 이해하는 데 꼭 필요한 개념들입니다.

- **2장**에서는 Samplomatic을 쓰는 데 필요한 모든 것을 다룹니다. 회로를 박스(box)로 묶는 법, 층별로 노이즈를 학습하는 법, 그리고 *Executor* 프리미티브로 작업을 실행하는 법을 배웁니다.

- 마지막으로 **3장**에서는 Samplomatic을 활용하는 두 가지 Qiskit 애드온 [PNA, Propagated Noise Absorption)](https://qiskit.github.io/qiskit-addon-pna/)(PNA)와 [SLC, Shaded Lightcones](https://qiskit.github.io/qiskit-addon-slc/)(SLC)을 소개합니다. 

**사용량 안내:** *Heron 장치 기준 총 예상 QPU 사용 시간은 1분 30초입니다. 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션(calibration), 런타임 세션 지연까지 더하면 실제로는 더 오래 걸릴 수 있습니다.*

이 랩은 실제 QPU 하드웨어에서의 실행이 필요합니다. 고급 오류 완화를 위해 장치의 노이즈를 직접 학습할 텐데, 노이즈 학습(noise learning) 작업은 모두 실제 하드웨어에서만 돌아갑니다. 시뮬레이터로는 실행할 수 없습니다!

**Windows 사용자 참고:**
3장에서 사용하는 SLC 애드온은 파이썬 기반 화학 시뮬레이션 라이브러리인 `pyscf` 패키지에 의존하는데, 이 패키지는 Windows를 기본 지원하지 않습니다. Windows 사용자라면 qBraid, Google Colab 같은 클라우드 서비스를 이용하거나, WSL(Windows Subsystem for Linux)로 파이썬 환경을 만드는 것을 권장합니다.

## 설정 및 임포트
아래 `pip install` 줄의 주석을 해제하고 셀을 실행해 필요한 라이브러리를 설치하세요. 설치가 끝나면 잊지 말고 커널을 재시작하세요!

In [ ]:
# Install required packages
#!pip install -q -U 'qiskit[visualization]' qiskit-ibm-runtime
#!pip install -q matplotlib numpy ipython
#!pip install -q samplomatic
#!pip install -q qiskit-addon-utils qiskit-addon-pna qiskit-addon-slc
#!pip install --upgrade qc-grader
#!pip install plotly
#!pip install pylatexenc
#!pip install nbformat

In [ ]:
### Scientific Computing
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

### IPython
from IPython.display import display

### Qiskit 
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

### Qiskit IBM Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, Executor, QuantumProgram
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.options_models.noise_learner_v3_options import NoiseLearnerV3Options
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

### Samplomatic
import samplomatic
from samplomatic import Twirl, InjectNoise, ChangeBasis, build
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions, get_annotation

### Qiskit Addons
from qiskit_addon_utils.exp_vals.measurement_bases import get_measurement_bases
from qiskit_addon_utils.exp_vals.expectation_values import executor_expectation_values
from qiskit_addon_utils.noise_management import trex_factors, gamma_from_noisy_boxes
from qiskit_addon_pna import generate_noise_mitigating_observable
from qiskit_addon_slc.bounds import compute_backward_bounds, compute_forward_bounds, compute_local_scales, merge_bounds
from qiskit_addon_slc.utils import generate_noise_model_paulis, map_modifier_ref_to_ref
from qiskit_addon_slc.visualization import draw_shaded_lightcone


# Grader imports
from qc_grader.challenges.qgss_2026 import (
    grade_lab3_ex1,
    grade_lab3_ex2,
    grade_lab3_ex3,
    grade_lab3_ex4,
    grade_lab3_ex5,
)

# grader function to check your progress
from qc_grader.challenges.qgss_2026 import check_progress

#### 백엔드 정의:

이 랩에서는 **Heron 장치**를 사용합니다. 위에서 말했듯 백엔드에서 실행하는 노이즈 학습 작업은 시뮬레이터로 돌릴 수 없습니다.

튜토리얼 내내 *같은 백엔드*를 유지하는 것이 좋습니다. 백엔드의 노이즈를 학습하고 그 노이즈 모델로 고급 오류 완화 기법을 구현할 것이기 때문에, 중간에 백엔드를 바꾸면 그동안 학습해 둔 노이즈 모델을 새 백엔드에 쓸 수 없게 됩니다.

참고: 백엔드 연결에 문제가 있다면 Lab 0으로 돌아가서 IBM Quantum Platform 계정이 올바르게 설정되어 있는지 확인하세요.

In [ ]:
service = QiskitRuntimeService()

# Define the backend: 
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True,
    filters=lambda b: b.processor_type.get('family') == 'Heron'
)

print(f"Backend     : {backend.name}")
print(f"# qubits    : {backend.num_qubits}")
print(f"Basis gates : {backend.basis_gates}")

# Create ISA pass manager for transpilation with optimization_level=0
isa_pm = generate_preset_pass_manager(backend=backend, optimization_level=0)


## 0장 — 오늘날 양자 하드웨어의 오류

오늘날의 양자 하드웨어는 노이즈를 지니고 있습니다. 게이트 하나, 유휴 시간 한 순간, 측정 한 번마다 작은 오류가 생겨나고, 유틸리티 규모(utility scale, 100큐비트 이상의 회로)에서는 이 오류가 더는 무시할 수 없는 수준으로 쌓입니다. 완전한 결함 허용(fault tolerance)이 실현되기 전까지, 이런 장치에서 *유의미한* 답을 얻는 유일한 방법은 노이즈를 관리하는 것입니다. 억제할 수 있는 곳에서는 억제하고, 그럴 수 없는 곳에서는 특성을 파악하고, 결과를 읽어낼 때는 노이즈가 만드는 편향(bias)을 완화해야 합니다. 그래서 노이즈를 다루는 일은 오늘날의 장치에서 유용한 결과를 얻는 과정의 한가운데에 있고, 그 기법들은 지금도 활발히 개발되고 있습니다.

노이즈를 다루는 기법은 크게 네 갈래로 나뉩니다. 갈래마다 *언제* 작동하는지, *어떤* 비용이 드는지, 그리고 오류를 고치는지, 표시만 하는지, 아니면 사후에 통계만 보정하는지가 서로 다릅니다. 이 랩의 중심이 될 새 도구를 소개하기 전에 네 가지를 한 번씩 훑어볼 만한 이유입니다.

- **오류 정정**(Error correction, EC)은 논리 정보를 중복해서 인코딩하고 신드롬(syndrome)을 반복 추출하며, 결함 허용 방식으로 구현하면 논리 오류율을 물리 오류율보다 낮게 억누를 수 있습니다. 다만 완전한 양자 오류 정정(QEC)은 큐비트와 시간 오버헤드가 커서 단기 장치에서는 감당하기 어렵습니다.

- **오류 검출**(Error detection, ED)은 오염됐을 가능성이 높은 실행을 표시해 사후 선택(post-selection)으로 버리고, 더 작지만 더 깨끗한 표본을 남깁니다. 이 방법의 일부 원리는 검출 전용 QEC 부호에서 나온 것일 수 있습니다. 부호의 거리(distance)가 오류의 발생은 알리지만 오류를 특정해 되돌릴 만큼의 정보는 주지 못하는 경우로, 이렇게 보면 ED는 오류 정정의 극한 사례인 셈입니다. 계층적 방식에서는 ED 덕분에 상위 부호가 표시된 오류를 *소거*(erasure)로 취급할 수 있습니다(위치를 아는 오류는 모르는 오류보다 훨씬 고치기 쉽습니다). 부호 없이 더 값싼 검사로도 가능합니다. 이상적인 회로라면 보존해야 할 대칭성(보존되는 패리티나 입자 수 등)을 확인해서, 이를 위반한 샷(shot)을 버리는 식입니다. 어느 쪽이든 ED는 쓸 수 있는 표본이 줄어드는 대신, 결함 허용 EC보다 훨씬 낮은 구현 오버헤드로 작동합니다.

- **오류 완화**(Error mitigation, EM)는 개별 샷을 고치려 들지 않습니다. 대신 노이즈 섞인 실행을 여러 번 반복해 얻은 통계에 편향을 주거나 가중치를 다시 매겨서, 기댓값(또는 분포)의 *추정치*가 이상적인 값에 가까워지도록 만듭니다([Cai et al., *Quantum error mitigation*, Rev. Mod. Phys. 95, 2023](https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.95.045005)). 오늘날의 유틸리티 규모 실험이 작동하는 영역이 바로 여기이고, 이 랩의 초점이기도 합니다.

- **오류 억제**(Error suppression, ES)는 실행 중에 작동해 노이즈 자체의 모양을 바꿉니다. 유휴 큐비트에 펄스를 삽입해 결맞음 오류(coherent error)를 상쇄하는 동역학적 디커플링(dynamical decoupling, DD), 게이트 드레싱을 무작위로 바꿔 잔여 오류를 확률적 파울리 채널(stochastic Pauli channel)로 만드는 파울리 트월링(Pauli twirling, PT)이 그 예입니다([Wallman & Emerson, PRA 94, 2016](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)).

Qiskit Runtime 프리미티브를 써 본 적이 있다면, 이 넷 중 둘은 눈치챘든 아니든 이미 사용해 본 셈입니다. `Estimator`가 구현하는 것이 바로 오류 억제와 오류 완화입니다(오류 정정과 검출은 프리미티브가 대신해 주는 영역이 아닙니다). `Estimator`는 기본 옵션에서도 체계적인 판독(readout) 편향을 이미 보정하고 있습니다(Twirled Readout Error eXtinction, TREX). 1.1.2절에서 보겠지만, 여기서 더 나아가 `EstimatorOptions`로 오류 억제·완화 기법을 직접 추가로 켤 수도 있습니다. 이 옵션들의 공통점은 *회로 전체*에 하나의 정책으로 적용된다는 것, 그래서 실행마다 정해진 형태의 오버헤드가 더해진다는 것입니다.

이 랩에서는 Qiskit Runtime과 나란히 사용해 그 *회로 전체* 설정을 넘어설 수 있게 해 주는 도구(*Samplomatic*)를 소개합니다. 이 도구가 있으면 노이즈를 층 단위로, 관측량(observable) 단위로 관리할 수 있습니다. 1장에서는 먼저 익숙한 `EstimatorOptions`가 무엇을 하고 어떤 비용이 드는지 짚어 보는 것으로 시작합니다.

## 1장 — Runtime 제품으로서의 오류 완화, 그리고 그 비용 뒤에 있는 이론

이 장에서는 익숙한 Runtime 옵션을 하나씩 열어 봅니다.

- 1.1절에서는 `Estimator`가 제공하는 오류 억제·오류 완화 기법을 차례로 살펴보고, 각 기법을 직접 손으로 설정해 봅니다.

- 1.2절에서는 스위치보다 한 단계 *아래*로 내려가서, 이 기법들이 딛고 서 있는 가장 기본적인 원리인 *드레싱*(dressing)과 *교환 관계*(commutation)라는 작은 대수 조각을 다룹니다.

- 1.3절에서는 회로 전체에 일괄 적용되는 이 스위치들이 *하지 못하는* 일은 무엇인지, 자원 오버헤드로 치르는 비용은 얼마인지 살펴보며 장을 마무리합니다. 랩의 나머지에서 다룰 박스 단위의 더 정밀한 도구가 왜 필요한지를 이곳에서 알수있게 됩니다.

## 1.1  클라우드 서비스로서의 오류 완화

서론에서 Qiskit Runtime이 오류 완화를 하나의 제품으로 제공한다고 했습니다. 켜기만 하면 되는 옵션들의 모음이고, 회로 전체에 단일 정책으로 적용됩니다. 이 절에서는 `EstimatorV2` 프리미티브의 그 옵션들을 정리합니다. 각 옵션이 무엇을 하고 어떤 `resilience_level` 프리셋이 그것을 켜는지 살펴본 다음(1.1.1절), 기법별 설정과 동작을 자세히 들여다봅니다(1.1.2절). 절의 끝에 있는 Exercise 1에서는 다섯 가지 기법을 각각 별도의 `EstimatorOptions`에 설정해서 기법별 효과를 분리해 볼 수 있게 합니다. 이 기법들의 바탕이 되는 대수는 1.2절에서 다룹니다.

### 1.1.1  Runtime 프리미티브의 옵션들

`EstimatorV2` 프리미티브는 오류 억제·완화 기법들을 `EstimatorOptions`라는 객체 하나 뒤에 묶어 제공합니다. 공식 목록은 [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques)에서 살펴볼 수 있습니다. `resilience_level` 옵션은 정해진 조합을 한꺼번에 켜 주는 편의용 프리셋입니다. 아래 표에 각 기법과 그 역할, 어느 프리셋 레벨에서 켜지는지를 정리했습니다. 정확한 속성 경로는 1.1.2절에서 소개합니다.

| 기법 | 역할 | 한 줄 설명 | 활성화 방법 |
|---|---|---|---|
| **DD**(동역학적 디커플링) | 억제 | 유휴 큐비트에 펄스 시퀀스를 삽입해, 큐비트가 대기하는 동안 쌓이는 결맞음 오류를 상쇄 | 직접 켜기 |
| **PT**(파울리 트월링, 게이트) | 억제 | 2큐비트 게이트 노이즈의 결맞음 성분을 사실상 확률적 파울리 채널로 다듬어, 후속 기법(PEA, PEC)이 파울리 채널 노이즈를 가정할 수 있게 함 | 레벨 2 |
| **TREX**(트월링 기반 판독 오류 제거) | 완화 | 무작위 X 게이트로 측정을 트월링해 판독 오류 전달 행렬을 대각화하고, 스케일 재조정만으로 역변환할 수 있게 함 | 레벨 1(기본값) |
| **ZNE**(제로 노이즈 외삽) | 완화 | 노이즈를 여러 배율로 증폭한 뒤(예: 게이트 폴딩) 기댓값을 노이즈 0 극한으로 외삽 (비편향 보장은 없음) | 레벨 2 |
| **PEA**(확률적 오류 증폭) | 완화 | ZNE의 게이트 폴딩 증폭을 *학습된* 파울리-린드블라드 노이즈 모델 기반 증폭으로 바꿔 노이즈를 더 정확하게 스케일링 (외삽 단계는 ZNE 그대로) | 직접 켜기(ZNE와 함께) |
| **PEC**(확률적 오류 상쇄) | 완화 | 앙상블 평균이 학습된 노이즈 채널을 역변환하도록 반노이즈 회로를 샘플링해 비편향 추정을 얻음. 대가는 샘플링 오버헤드 $\gamma^2$ | 직접 켜기 |

기본적으로 Runtime은 **resilience level=1**로 동작하며, 이때 켜지는 것은 TREX뿐입니다. 레벨별로 무엇이 켜지는지는 [Configure error mitigation](https://docs.quantum.ibm.com/guides/configure-error-mitigation) 가이드를 참고하세요. 다음 절에서 기법별 옵션과 동작을 자세히 살펴봅니다.

### 1.1.2  기법별 상세

이 절에서는 1.1.1절 표의 기법을 하나씩 풀어 봅니다. 무엇을 하는지, 어떤 옵션으로 제어하는지, 언제 도움이 되는지를 정리합니다. 그림과 전체 설명은 공식 가이드 [Error mitigation and suppression techniques](https://docs.quantum.ibm.com/guides/error-mitigation-and-suppression-techniques)에서 가져왔습니다.

#### 동역학적 디커플링(DD) — *억제*
유휴 큐비트는 다른 큐비트의 작업이 끝나기를 기다리는 동안 결맞음 오류를 얻습니다. DD는 그런 유휴 큐비트에, 전부 곱하면 항등 연산이 되는 펄스 시퀀스를 삽입합니다. 의도한 계산은 그대로 유지되면서, 천천히 쌓이는 결맞음 드리프트는 펄스들에 의해 평균적으로 지워집니다. 회로에 유휴 구간이 많을 때 가장 효과적이고, 빽빽하게 채워진 회로에서는 별 효과가 없거나 오히려 노이즈를 키워버릴 수도 있습니다.

![동역학적 디커플링: 큐비트의 유휴 구간에 삽입된 XX 펄스 시퀀스 (IBM Quantum 문서).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/dd.avif)

- `dynamical_decoupling.enable` — DD를 켭니다(기본값은 꺼짐).
- `dynamical_decoupling.sequence_type` — 사용할 펄스 시퀀스. 기본값은 `"XX"`이고, `"XpXm"`, `"XY4"` 등도 있습니다.

#### 파울리 트월링(PT) — *억제*
트월링(랜덤화 컴파일링)은 선택한 게이트의 앞뒤를 무작위 단일 큐비트 파울리로 감싸되, 이상적인 연산은 변하지 않도록 파울리를 고릅니다. 무작위 인스턴스를 많이 만들어 평균하면 임의의 노이즈 채널이 확률적 *파울리* 채널이 됩니다. 깊이에 따라 제곱으로 쌓이던 결맞음 오류가, 선형으로만 자라는 파울리 노이즈로 다듬어지는 것입니다. 하드웨어 오류 대부분이 2큐비트 게이트에서 나오므로 트월링은 보통 네이티브 2큐비트 게이트에 적용하며, 파울리 채널 노이즈를 가정하는 PEA와 PEC의 전제 조건이기도 합니다.

![파울리 트월링은 회로 하나를 동등한 트월링 회로들의 무작위 앙상블로 대체합니다 (IBM Quantum 문서).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/pauli_twirling.svg)

- `twirling.enable_gates` — 2큐비트 게이트 층을 트월링합니다.
- `twirling.num_randomizations` — 무작위 트월링 인스턴스의 수.
- `twirling.shots_per_randomization` — 인스턴스당 샷 수.

#### 트월링 기반 판독 오류 제거(TREX) — *완화*
TREX가 겨냥하는 것은 *측정* 오류입니다. 측정 직전에 무작위로 X를 삽입하고 측정 후에 고전 비트를 도로 뒤집는 방식으로 판독을 트월링합니다. 판독 오류가 있는 상황에서 이렇게 하면 판독 오류 전달 행렬이 대각화되어, 간단한 스케일 재조정만으로 역변환할 수 있게 됩니다. 재조정 계수는 0 상태로 준비한 무작위 회로를 벤치마킹해서 학습하는데, 캘리브레이션 회로 몇 개가 그 비용입니다. 다섯 기법 중 유일하게 기본으로 켜져 있는 방법입니다(*resilience level 1*).

![측정 트월링: 측정 앞의 X 게이트와 측정 뒤의 고전 비트 반전은, 노이즈가 없다면 일반 측정과 동등합니다 (IBM Quantum 문서).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/measurement_twirling.svg)

- `resilience.measure_mitigation` — TREX를 켭니다.
- `resilience.measure_noise_learning.num_randomizations`, `...shots_per_randomization` — 측정 노이즈 학습을 제어합니다.

#### 제로 노이즈 외삽(ZNE) — *완화*
ZNE는 노이즈를 여러 단계로 *증폭*해 가며 회로를 실행한 뒤, 기댓값을 노이즈 0 지점으로 외삽합니다. Runtime은 디지털 게이트 폴딩으로 노이즈를 증폭합니다. 유니터리 $U$를 $U U^\dagger U$(또는 더 긴 폴딩)로 바꿔 노이즈를 정해진 배수만큼 키우고, 결과를 선택한 함수 형태로 피팅하는 방식입니다. 정확도가 좋아지는 경우가 많지만 비편향이 *보장되지는* 않습니다. 기본 설정은 세 가지 노이즈 배율을 샘플링하므로 대략 3배의 오버헤드가 듭니다.

![제로 노이즈 외삽: 디지털 게이트 폴딩으로 노이즈를 증폭하고, 결과를 노이즈 0 극한으로 외삽합니다 (IBM Quantum 문서).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/zne.avif)

- `resilience.zne_mitigation` — ZNE를 켭니다.
- `resilience.zne.noise_factors` — 증폭 배율(기본값 `(1, 3, 5)`).
- `resilience.zne.extrapolator` — 피팅 형태(예: 선형, 지수).

#### 확률적 오류 증폭(PEA) — *완화*
PEA는 ZNE를 *위한* 더 정확한 증폭기입니다. 게이트 폴딩 대신, 먼저 각 얽힘 층의 트월링된 파울리-린드블라드 노이즈 모델을 **학습**한 다음, 그 학습된 모델에서 뽑은 단일 큐비트 노이즈를 확률적으로 주입해서 증폭합니다. 합성으로 만든 증폭이 아니라 실제 장치 노이즈를 충실히 스케일링하는 것입니다. 증폭된 결과는 그대로 ZNE의 외삽 단계로 들어가므로, PEA는 ZNE 위에 얹어서 켭니다. 유틸리티 규모 실험에서는 대개 가장 좋은 증폭기 선택지입니다.

- `resilience.zne_mitigation = True`와 함께 `resilience.zne.amplifier = "pea"`를 설정합니다.

#### 확률적 오류 상쇄(PEC) — *완화*
PEC도 노이즈 모델을 학습하지만, 증폭하는 대신 *역변환*합니다. 평균이 역노이즈 채널을 구현하는 "반노이즈(anti-noise)" 회로의 앙상블을 샘플링해서 **비편향** 추정을 얻습니다. 그 대가는 $\gamma^2$으로 스케일링되는 샘플링 오버헤드로, 회로 깊이와 총 노이즈가 커질수록 빠르게 불어납니다. PEC가 적당한 크기의 회로에서만 실용적인 이유입니다(이 비용은 1.3절에서 다시 다룹니다). 3장에서는 *Samplomatic*으로 PEC를 구현하고, PEC의 진화형인 음영 라이트콘(SLC)을 소개합니다.

- `resilience.pec_mitigation` — PEC를 켭니다.
- `resilience.pec.max_overhead` — 샘플링 오버헤드 $\gamma^2$의 상한. 상한을 넘길 것 같으면 Runtime이 물러섭니다.

**참고**: ZNE/PEA와 PEC는 노이즈를 증폭하느냐 상쇄하느냐로 서로 *반대되는* 전략이라서, **같은** `EstimatorOptions`에 함께 켤 수 없습니다.

<div class="alert alert-block alert-success">

**Exercise 1 — 각 기법 설정하기.**

1.1.2절의 기법과 속성 경로를 이용해 다섯 가지 기법을 각각 별도의 `EstimatorOptions` 객체에 설정한 뒤, 이들을 `"dd"`, `"pt"`, `"trex"`, `"zne"`, `"pec"` 키를 가진 딕셔너리 `options_dict`로 모으세요. 각 기법은 따로따로 검사되며, 다섯 개가 모두 맞아야 통과입니다. 틀린 항목이 있으면 채점기(grader)가 알려 줍니다.

- **`dd_options`** — 동역학적 디커플링: `enable = True`, `sequence_type = "XY4"`
- **`pt_options`** — 파울리 트월링: `enable_gates = True`, `strategy = "active"`, `num_randomizations = 32`
- **`trex_options`** — TREX: `measure_mitigation = True`
- **`zne_options`** — PEA를 쓰는 ZNE: `zne_mitigation = True`, `noise_factors = [1, 3, 5]`, `amplifier = "pea"`
- **`pec_options`** — PEC: `pec_mitigation = True`, `max_overhead = 100`

</div>

In [ ]:
dd_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

pt_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

trex_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

zne_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

pec_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

options_dict = {
    "dd": dd_options,
    "pt": pt_options,
    "trex": trex_options,
    "zne": zne_options,
    "pec": pec_options,
}


In [ ]:
# grade your answer
grade_lab3_ex1(options_dict)

## 1.2  수학적 기반: 드레싱된 층과 교환 관계

1.1절의 옵션들은 회로 전체에 **한 번** 적용됩니다. 그런데 그중 여러 기법이 공통된 기본 연산 하나를 공유합니다. 게이트의 양옆을 단일 큐비트 파울리로 감싸되, 게이트의 논리적 동작은 그대로 두고 게이트에 붙은 노이즈만 변환되도록 파울리를 고르는 것입니다. 게이트 트월링은 이 연산을 직접 사용하고, ZNE/PEA와 PEC는 학습하는 파울리 노이즈 모델을 통해 간접적으로 사용합니다. 이렇게 감싼 게이트를 **드레싱된**(dressed) 게이트라고 부릅니다.

이 절에서는 *게이트를 드레싱한다*는 것이 무슨 뜻인지 설명합니다. 1.1절의 오류 완화 스위치들이 왜 그런 식으로 동작하는지 이해하는 데 도움이 될 것입니다. 그리고 이것은 Samplomatic 프레임워크의 박스 단위 도구들이 쓰는 연산이기도 합니다. 지금까지 본 *회로 전체 스위치* 방식을 이 프레임워크가 어떻게 넘어서는지는 2장에서 보게 됩니다.

**드레싱된 층**(dressed layer)이란 양옆이 단일 큐비트 유니터리로 감싸인 2큐비트 유니터리를 말합니다. 먼저 일반적인 형태로 소개한 다음, CZ를 파울리로 드레싱하는 경우로 좁힙니다. 이후에 다룰 모든 2큐비트 층이 바로 이 설정입니다. 절 전체를 관통하는 실마리는 파울리와 원래 게이트 사이의 *교환 관계*(클리포드 켤레 연산)입니다. 드레싱 표에 구조를 만들어 주는 것이 이 관계이고, 나중에 Samplomatic이 박스 단위로 가상 파울리를 회로에 전파할 때 쓰는 연산도 바로 이것입니다.

### 1.2.1  유니터리 드레싱하기

임의의 $n$큐비트 유니터리 $U$에 대해, *드레싱된* 버전은

$$\tilde{U} \;=\; V_\text{out}\, U\, V_\text{in},$$

입니다. 여기서 $V_\text{in}$과 $V_\text{out}$은 (큐비트마다 하나씩 걸리는) 단일 큐비트 유니터리들의 곱입니다. 드레싱이 전역 위상을 제외하고 $V_\text{out}\,U\,V_\text{in} = U$를 만족하면 *불변*(invariant) 드레싱이라고 부릅니다. $U$의 논리적 동작은 그대로지만, $U$에 붙어 있는 노이즈 채널은 $V_\text{in}$과 $V_\text{out}$에 의해 *트월링*됩니다([Wallman & Emerson, *Noise tailoring for scalable quantum computation via randomized compiling*, PRA 94, 2016](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)). 자연스럽게 두 가지 질문이 따라옵니다. 어떤 쌍 $(V_\text{in}, V_\text{out})$이 $U$를 불변으로 두는가, 그리고 그런 쌍을 체계적으로 어떻게 나열하는가.

랩 전체에서 다룰 경우는 $V_\text{in}$과 $V_\text{out}$이 파울리이고 $U$가 클리포드인 상황입니다. IBM 하드웨어의 2큐비트 게이트(CZ, CX, ECR)가 바로 이 설정에 해당합니다. `qiskit.quantum_info`의 도구들이 이를 쉽게 확인하도록 해줄것입니다. `Pauli("IX").evolve(U, frame='s')`는 부호까지 정확한 $U\,P\,U^\dagger$를 돌려주고, `Pauli.equiv(other)`는 두 파울리를 전체 위상을 무시하고 비교합니다. 우리에게 필요한 "전역 위상을 제외하고 불변"이라는 개념 그대로입니다.

In [ ]:
def dress(U: QuantumCircuit, v_in: str, v_out: str) -> QuantumCircuit:
    """Build the circuit  V_out . U . V_in  as a QuantumCircuit.
    """
    n = U.num_qubits
    assert not v_in.startswith(("+", "-", "i")), (
        f"v_in must be an unsigned Pauli string, got {v_in!r}"
    )
    assert not v_out.startswith(("+", "-", "i")), (
        f"v_out must be an unsigned Pauli string, got {v_out!r}"
    )
    assert len(v_in) == len(v_out) == n, "Pauli string length must match U's num_qubits"
    qc = QuantumCircuit(n)
    # Qiskit string 'q_{n-1}...q_0' — reverse to iterate in qubit order.
    for q, p in enumerate(reversed(v_in)):
        if p != "I":
            getattr(qc, p.lower())(q)
    qc.compose(U, inplace=True)
    for q, p in enumerate(reversed(v_out)):
        if p != "I":
            getattr(qc, p.lower())(q)
    return qc


def is_invariant(U: QuantumCircuit, v_in: str, v_out: str) -> bool:
    """True iff  V_out . U . V_in  equals  U  up to a global phase.

    Takes *unsigned* Pauli strings v_in, v_out. The invariance condition is
    U V_in U† = ±V_out; the ± becomes a global phase on the dressed operator,
    so it does not affect invariance. We use `Pauli.equiv`, which is
    phase-insensitive, to compare.
    """
    return Pauli(v_in).evolve(U, frame='s').equiv(Pauli(v_out))

### 1.2.2  CZ 예제

$U = \mathrm{CZ}$로 좁히는 이유는 랩 전체에서 쓰는 2큐비트 게이트가 CZ이기 때문입니다. 2.6.1절의 이징 회로도 전부 CZ 게이트로 만들어져 있습니다. 특정 파울리 드레싱이 CZ를 불변으로 둔다는 사실을 직접 계산으로 확인한 다음, 16개의 2큐비트 파울리를 전부 CZ에 통과시켜 전체 맵 $V_\text{in} \mapsto V_\text{out}$을 만듭니다. 각 $V_\text{in}$마다 (부호를 제외하면) 유효한 $V_\text{out}$은 정확히 하나입니다.

아래에서 출력할 표에는 서로 다른 두 가지 정보가 함께 들어 있는데, 이 둘을 섞어서 생각하면 안 됩니다. *드레싱 맵*은 $V_\text{out}$으로 실제 적용할 **부호 없는** 파울리를 기록하며, `dress()`가 소비하는 것이 이것입니다.

**대수적 부호**, 즉 $\mathrm{CZ}\cdot V_\text{in}\cdot \mathrm{CZ}$가 $+V_\text{out}$과 같은지 $-V_\text{out}$과 같은지는 게이트 불변성에는 영향을 주지 **않습니다**(부호는 전역 위상으로 흡수됩니다). 하지만 1.2.3절의 전파 규칙에서는 부호가 **중요해집니다**.

표 자체에는 깔끔한 구조가 있다는 것도 보게 될 겁니다. $Z$ 성분은 CZ를 그대로 통과하고, 한 큐비트의 $X$나 $Y$ 성분은 **반대쪽** 큐비트에 $Z$를 하나 얻으며, $XY \leftrightarrow YX$ 쌍에는 마이너스 부호가 붙습니다. 아래 출력의 맨 오른쪽(부호) 열에서 확인할 수 있습니다.

In [ ]:
# Bare CZ layer
cz_layer = QuantumCircuit(2, name="CZ")
cz_layer.cz(0, 1)
cz_layer.draw("mpl")

In [ ]:
# apply X on qubit 0 before CZ, and X on qubit 0 + Z on qubit 1 after.

print("V_in='IX', V_out='ZX' leaves CZ invariant?",
      is_invariant(cz_layer, "IX", "ZX"))

In [ ]:
# Build two related objects from the Pauli algebra:
#   cz_twirl_map       : V_in -> V_out  (UNSIGNED — what to apply as a dressing)
#   cz_commutation_map : V_in -> (sign, V_out)  (signed — for propagation rules)

PAULIS = ["I", "X", "Y", "Z"]


def split_sign(label: str) -> tuple[int, str]:
    """Split a signed Pauli label into (+/-1, unsigned_label)."""
    if label.startswith("-"):
        return -1, label[1:]
    if label.startswith("+"):
        return +1, label[1:]
    return +1, label


cz_twirl_map       = {}
cz_commutation_map = {}

for p1 in PAULIS:         # Pauli on qubit 1 (leftmost in string)
    for p0 in PAULIS:     # Pauli on qubit 0 (rightmost in string)
        v_in   = p1 + p0
        signed = Pauli(v_in).evolve(cz_layer, frame="s").to_label()
        sign, bare = split_sign(signed)
        cz_twirl_map[v_in]       = bare
        cz_commutation_map[v_in] = (sign, bare)

print(f"Total Pauli dressings for CZ: {len(cz_twirl_map)}  (expected 16)")
print()
print(f"  {'V_in':<6}{'V_out (apply)':<16}{'algebraic sign'}")
print(f"  {'-'*6}{'-'*16}{'-'*14}")
for v_in, v_out in cz_twirl_map.items():
    sign, _ = cz_commutation_map[v_in]
    sign_str = "+" if sign > 0 else "−"
    print(f"  {v_in:<6}{v_out:<16}{sign_str}")

### 1.2.3  교환 관계: 파울리는 클리포드를 어떻게 통과하는가

1.2.2절 표의 구조는 우연이 아닙니다. 임의의 파울리 $P$와 클리포드 $U$에 대해

$$U\,P\,U^\dagger \;=\; \pm\,P',$$

이고, 여기서 $P'$은 또 다른 파울리입니다. 이것이 클리포드 군을 정의하는 성질입니다. $V_\text{in}$이 주어졌을 때 $V_\text{out}$을 찾는 일은, $V_\text{in}$을 $U$에 통과시켜 전파한 뒤 부호까지 포함해서 결과를 읽어내는 것과 같습니다. 2장에서 Samplomatic이 가상 파울리를 게이트 박스 너머로 전파할 때마다 쓰는 연산이 바로 이것입니다. Qiskit에서는 `Pauli.evolve(U, frame='s')`가 이를 구현하고, `.to_label()`은 `"-YX"`처럼 부호가 붙은 파울리 문자열을 돌려줍니다.

같은 방식으로 CX를 확인해보면, 구조는 비슷하지만 CX의 제어/타깃 역할에서 오는 비대칭이 있는 표가 나옵니다.

In [ ]:
# The same Pauli.evolve(..., frame="s") helper, applied to two different gates.
# The propagation rule is fixed by the gate, so CZ and CX give different tables.
cx_layer = QuantumCircuit(2, name="CX")
cx_layer.cx(0, 1)  # control = q0, target = q1

reps = ["IX", "XI", "IZ", "ZI", "IY", "YI"]

print("CZ . P . CZ  (symmetric in the two qubits):")
for p in reps:
    print(f"  CZ . {p} . CZ  =  {Pauli(p).evolve(cz_layer, frame='s').to_label()}")

print("\nCX . P . CX  (control = q0, target = q1):")
for p in reps:
    print(f"  CX . {p} . CX  =  {Pauli(p).evolve(cx_layer, frame='s').to_label()}")

위의 두 표가 서로 다른 이유는 전파 규칙이 게이트에 의해 정해지기 때문입니다. CZ는 대칭적입니다. 어느 큐비트에 있든 $Z$는 그대로 통과하고, $X$나 $Y$는 반대쪽 큐비트에 $Z$를 하나 얻습니다. CX는 그렇지 않습니다. 타깃 큐비트의 $X$와 제어 큐비트의 $Z$는 두 큐비트로 퍼지지만, 제어 큐비트의 $X$와 타깃 큐비트의 $Z$는 변하지 않고 통과합니다.

절차는 두 경우 모두 같습니다. 파울리를 게이트로 켤레화하고, 부호가 붙은 결과를 읽어내면 됩니다. 1.1.2절의 `twirling.enable_gates` 스위치부터 2장에서 소개할 Samplomatic의 박스 단위 `Twirl` 어노테이션까지, 이 랩의 트월링 기반 기법은 전부 이 연산 위에 세워져 있습니다.

그 기법들을 가르는 것은 연산이 적용되는 규모입니다. `EstimatorOptions` 스위치는 이 연산을 회로 전체에 한 번 적용합니다. 가장 기본적인 방식이지만, 층이나 관측량마다 다르게 다뤄야 하는 완화 작업에는 통하지 않습니다. 1.3절에서는 Runtime 옵션이 할 수 있는 일과 할 수 없는 일, 그리고 그 자원 비용을 정리합니다. 랩의 나머지에서 더 정밀한 도구를 쓰는 이유가 여기서 나옵니다.

## 1.3  구조를 아는 완화 전략이 필요한 이유

앞의 절들에서는 `EstimatorOptions` 스위치가 할 수 있는 일과 그 밑에 있는 대수를 살펴봤습니다. 이 절에서는 회로 전체 스위치가 어디서 부족한지, 그것으로는 표현할 수 없는 오류 완화 작업이 무엇인지, 회로가 커질수록 비용이 어떻게 불어나는지를 봅니다. 그 부족함을 메우는 것이 랩의 나머지를 떠받치는 박스 단위 도구, Samplomatic입니다.

#### Runtime 옵션이 하지 못하는 일

Runtime 옵션이 제공하는 것은 "모든 유휴 구간에 DD 삽입"이나 "모든 2큐비트 게이트 트월링" 같은 **회로 전체 스위치**입니다. 각각은 회로 전역에 단일 정책으로 적용되고, 작은 워크로드 대부분에는 이것으로 충분합니다. PEA와 PEC는 내부적으로는 *얽힘 층마다* 노이즈 모델을 학습하지만, API가 층 단위 제어를 열어 주지 않습니다. 층의 학습된 노이즈를 들여다볼 수도, 덮어쓸 수도, 층마다 다른 전략을 적용할 수도 없는 것입니다. 아예 손이 닿지 않는 기능도 있습니다. 회로를 완화하는 대신 관측량 자체를 다시 써서 학습된 노이즈를 흡수하는 것인데, 3장에서 **전파 노이즈 흡수**(propagated noise absorption, PNA)로 만나게 됩니다.

2장에서는 이런 정밀한 제어를 제공하는 도구들을 소개합니다. `Twirl`, `InjectNoise`, `ChangeBasis` 같은 지시어를 어노테이션으로 붙인 회로 조각인 *박스*, 그리고 `Sampler`·`Estimator`의 박스 인지 대응물인 `NoiseLearnerV3`와 `Executor`입니다.

#### 숨은 비용: 샘플링 오버헤드

이 스위치들을 켜는 것은 공짜가 아니고, 청구서는 큐비트가 아니라 **샷**으로 날아옵니다. 오류 완화는 *공간* 오버헤드를 *시간* 오버헤드와 맞바꿉니다. (오류 정정처럼) 물리 큐비트를 더 쓰는 대신, 노이즈 섞인 회로를 여러 번 돌리고 후처리하는 것입니다. 이 비용의 일부는 옵션에서 바로 읽어낼 수 있는 단순한 배수입니다. ZNE의 `noise_factors`, `twirling.num_randomizations`, TREX의 판독 캘리브레이션 배치가 그렇습니다. 예를 들어 ZNE 배율 세 개에 랜덤화 32개면, 완화 없는 실행 한 번에 비해 벌써 `3 × 32` 규모의 변형 회로를 돌리는 셈입니다.

옵션에서 읽어낼 수 *없는* 비용이 유틸리티 규모 회로의 발목을 잡습니다. 확률적 기법의 **샘플링 오버헤드**는 *지수적으로* 자랍니다. PEC의 실행 시간 $J$는
$$J = \bar{\gamma}^{\,nd}\,\beta^{d},$$
로 주어지고, $n$은 큐비트 수, $d$는 깊이입니다. 밑에 있는 $\bar{\gamma}$는 *학습된* 노이즈 모델이 정합니다. 일종의 품질 지표라고 볼 수 있습니다. 게이트 오류가 낮을수록 $\bar{\gamma}$가 작아집니다. $\beta$는 회로 한 층당 걸리는 시간(속도 지표)입니다. $\bar{\gamma}$가 지수 안에 들어 있기 때문에, 게이트 오류를 낮춰서 $\bar{\gamma}$를 조금만 줄여도 $\bar{\gamma}^{nd}$의 크기는 극적으로 줄어듭니다.

<img src="https://research-website-prod-cms-uploads.s3.us.cloud-object-storage.appdomain.cloud/New_Fig_Gamma_Epsilon2_Tex_100q_trotter_a712254dca.jpg" width="450" alt="100큐비트 이징 트로터 회로에 대한 PEC 회로 오버헤드 추정">

*그림 1: 이징 스핀 사슬의 시간 전개를 나타내는 깊이 400 및 4000의 100큐비트 트로터화 회로에 대한 PEC 회로 오버헤드 추정. 빨간 점선은 1kHz 고정 샘플링 속도를 가정할 때 1일치 실행 시간에 해당합니다. K. Temme, E. van den Berg, A. Kandala, and J. Gambetta, "[Error mitigation is the path to quantum computing usefulness](https://www.ibm.com/quantum/blog/gammabar-for-quantum-advantage)," IBM Quantum blog, 2022. © IBM.*

위 그림은 이 랩에서 쓰는 것과 같은 종류의 회로, 즉 이징 해밀토니안 트로터 전개(2.6절에서 소개합니다)에 대한 것입니다. 2큐비트 게이트 오류가 낮아질수록 오버헤드가 몇 자릿수씩 떨어지는 것이 보입니다. 그런데 `EstimatorOptions`의 회로 전체 스위치로는 $\bar{\gamma}$를 결정하는 층별 노이즈를 들여다볼 수도, 손댈 수도 없습니다. 이 간극을 랩의 나머지가 메웁니다. 2장에서는 `NoiseLearnerV3`로 층별 노이즈를 *측정*하고, 3장에서는 층과 관측량을 하나하나 따로 다루는 방식(PNA와 SLC)으로 오버헤드를 *줄입니다*.

## 2장 — Samplomatic과 NoiseLearner

노이즈를 개별 층이나 게이트 수준에서 완화하려면 세 가지가 필요합니다. 회로의 각 부분을 따로 구분하는 방법, 각 부분의 노이즈 측정, 그리고 완화된 프로그램을 실행하고 결과를 모으는 방법입니다. 이 세 조각을 공급하는 것이 Samplomatic, `NoiseLearnerV3`, 그리고 `Executor` 프리미티브입니다. 셋이 모여 Qiskit Runtime의 [지시적 실행 모델(directed execution model)](https://quantum.cloud.ibm.com/docs/en/guides/directed-execution-model)을 이룹니다. 완화 의도를 클라이언트 쪽에서 기록하고, 회로 변형의 생성은 서버로 넘기는 모델입니다.

[Samplomatic](https://github.com/Qiskit/samplomatic)은 **박스**(box)와 **어노테이션**(annotation)으로 회로의 부분들에 이름을 붙여 구분합니다. 박스는 하나의 단위로 구획해 둔 회로 영역입니다. 게이트 하나일 수도, 층 하나일 수도, 마지막 측정일 수도 있습니다. 어노테이션은 박스에 붙이는 지시문입니다. `Twirl`은 박스의 드레싱을 무작위로 바꾸고, `InjectNoise`는 여기에 학습된 노이즈 채널이 적용될 것임을 선언하며, `ChangeBasis`는 측정 기저를 회전합니다. 지시가 회로 전체가 아니라 박스에 붙기 때문에 박스마다 다르게 다룰 수 있습니다. 이후 `build` 단계가 어노테이션이 붙은 회로를 파라미터화된 *템플릿*(template)과 *samplex* — 실행 시점에 박스 파라미터를 채우는 레시피 — 로 바꿔 줍니다.

측정을 담당하는 것이 [`NoiseLearnerV3`](https://quantum.cloud.ibm.com/docs/en/guides/noise-learning)입니다. 백엔드에서 특성화 회로를 돌리고, Samplomatic이 쓰는 것과 같은 박스 단위로 각 박스의 [파울리-린드블라드 노이즈 모델](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/utils-noise-learner-result-pauli-lindblad-error)을 반환합니다. 두 도구는 한 가지 패턴으로 맞물립니다. 박스가 `InjectNoise`로 "여기에 노이즈 모델이 들어온다"고 *선언*해 두면, `NoiseLearnerV3`가 측정한 바로 그 모델이 실행 시점에 `InjectNoise`의 참조를 따라 끼워지는 것입니다.

완화된 프로그램을 실제로 실행하는 것은 [`Executor`](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/executor) 프리미티브입니다. `NoiseLearnerV3`와 `Executor` 둘 다 백엔드에 작업을 제출하지만 목적이 다릅니다. `NoiseLearnerV3`는 *노이즈를 측정하기 위해* 회로를 돌리고, `Executor`는 템플릿과 samplex로 만든 박스 인지 프로그램을 돌려 *계산 결과를 얻습니다*. `Sampler`와 `Estimator`의 [박스 인지 대응물](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-runtime-primitives)인 셈입니다. **박싱**(boxing)은 이 세 단계 전체를 관통하는 구조입니다. 박스 단위 완화를 지정하는 방식도, 박스 단위 노이즈가 보고되는 방식도, Executor가 실행하는 대상도 모두 박스입니다. 그리고 3장의 애드온인 전파 노이즈 흡수(PNA)와 음영 라이트콘(SLC)이 층 하나, 관측량 하나 단위로 완화를 수행할 때 딛는 발판도 바로 이 박스 구조입니다.

이어지는 절들에서 작은 2큐비트 토이 모델로 각 도구를 하나씩 소개합니다. 그다음 2.6절에서 이 도구들을 1차원 이징 사슬에 모아서 적용해 봅니다. 3장에서도 계속 다룰 시스템입니다.

## 2.1  박스와 `Twirl` 어노테이션

회로를 박스로 묶는 것부터 시작합니다. 아주 단순한 예제로 출발해 봅시다. CZ 게이트 하나가 있는 2큐비트 회로입니다. `.box` 메서드로 CZ 게이트 둘레에 박스를 만들고, 이 박스에 `Twirl` 객체가 어노테이션으로 붙는다고 알려 줍니다.

1.1.2절에서 파울리 트월링을 오류 완화의 중요한 도구로 소개했습니다. 양자 회로를 파울리 트월링하면 회로의 노이즈가 *확률적 파울리 채널*로 다듬어집니다. 회로 하나를 무작위 회로들의 앙상블, 즉 *랜덤화*(randomization)로 대체하는 방식입니다. 짚고 넘어갈 점은, 트월링 자체가 회로의 오류를 없애 주는 것은 아니라는 사실입니다. 트월링의 역할은 다른 기법들이 노이즈를 효과적으로 완화할 수 있는 형태로 노이즈를 다듬는 것입니다.

`Twirl` 어노테이션은 그 박스를 *트월링*하라는 표시입니다.

In [ ]:
toy = QuantumCircuit(2)
with toy.box(annotations=[Twirl()]):
    toy.cz(0, 1)

toy.draw("mpl")

회로 다이어그램에는 CZ 게이트를 감싼 박스(빨간 사각형)가 보이지만, `Twirl` 어노테이션 자체는 그림에 나타나지 않습니다. 어노테이션은 박스에 붙은 메타데이터이기 때문입니다. 회로의 명령어들을 순회하면 어노테이션을 다시 읽어낼 수 있습니다:

In [ ]:
for idx, instruction in enumerate(toy):
    print(f"Box {idx}: annotations = {instruction.operation.annotations}")

위에 출력된 명령어들이 말해 주는 내용은 다음과 같습니다.

이 회로에는 박스가 하나 있고, 그 박스에 어노테이션이 하나 붙어 있습니다: `Twirl(group='pauli', dressing='left', decomposition='rzsx')`. 이것이 `Twirl`의 기본 설정입니다. 하나씩 뜯어보면:
- `group='pauli'`: 파울리 군 트월링을 적용합니다
- `dressing='left'`: 무작위 드레싱이 박스의 왼쪽에 붙습니다
- `decomposition='rzsx'`: 드레싱이 하드웨어 네이티브 게이트인 `rz`와 `sx`로 컴파일된다는 것을 기록합니다.

박스는 회로의 *어느* 부분을 트월링할지 표시하고, 어노테이션은 *어떻게* 할지를 기록합니다. 이 의도를 실행 가능한 회로로 바꾸는 것이 다음 단계입니다.

### 2.1.1  템플릿

Samplomatic 워크플로의 다음 단계는 템플릿 회로와 samplex를 *빌드*하는 것입니다. `build(boxed_circuit)` 메서드는 어노테이션이 붙은 회로를 두 객체로 바꿔 줍니다. 하나는 *템플릿*(임의의 랜덤화 하나를 실현할 수 있을 만큼 자유 게이트를 갖춘 파라미터화된 `QuantumCircuit`)이고, 다른 하나는 *samplex*(실행 시점에 그 파라미터들을 어떻게 채울지 기술한 레시피)입니다.

`build`를 쓸 때 흔히 겪는 오류가 하나 있는데, 드레싱이 갈 곳이 없는 경우입니다. 아래 셀에서 이를 재현해 봅니다.

In [ ]:
try:
    template_circuit, samplex = build(toy)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

`build`가 실패하는 것을 볼 수 있습니다. `Twirl` 어노테이션은 Samplomatic에게 무작위 파울리 드레싱을 박스의 *왼쪽*(입력) 면에 붙이라고 지시합니다. 그런데 박스의 논리적 동작이 변하지 않으려면 그 파울리가 CZ를 통과해 *오른쪽*(출력) 면으로 *가상 게이트*(virtual gate)가 되어 나와야 하고, 이후의 어떤 박스가 그것을 흡수해 줘야 합니다. 지금은 박스 뒤에서 회로가 그냥 끝나 버리기 때문에 오른쪽으로 나가는 가상 게이트들이 *내려앉을 곳이 없고*, 그래서 `SamplexBuildError`가 발생합니다. 오류 메시지는 큐비트 `[0, 1]`에 종결되지 않은 가상 게이트가 있다고 알려 줍니다. `Twirl` 박스에는 언제나 *컬렉터*(collector), 즉 가상 게이트를 받아 줄 또 다른 박스가 필요합니다. 다음 셀에서 측정 박스를 컬렉터로 추가합니다. 그러면 `build`가 성공적으로 실행됩니다.

In [ ]:
toy = QuantumCircuit(2)
with toy.box(annotations=[Twirl()]):
    toy.cz(0, 1)
with toy.box(annotations=[Twirl()]):
    toy.measure_all()

template_circuit, samplex = build(toy)
template_circuit.draw("mpl", fold=-1)

측정을 자기만의 `Twirl` 박스로 감싸 주면 오른쪽 드레싱들이 내려앉을 자리가 생깁니다. CZ 박스에서 나온 가상 게이트들이 측정 박스로 흡수되고, 측정 박스는 그와 동시에 판독을 트월링합니다. 이제 `build`가 성공해서 `template`과 `samplex`를 돌려줍니다.

템플릿의 회로 다이어그램을 보면 단일 큐비트 게이트들이 `rz`와 `sx`로 분해된 채 자유 파라미터로 남아 있습니다. `samplex`는 랜덤화를 하나 뽑을 때마다 그 파라미터들을 구체적인 무작위 파울리 할당으로 채우는 레시피를 담고 있습니다. 템플릿 하나와 samplex 하나가 랜덤화된 회로들의 앙상블 전체를 대신하는 것입니다.

### 2.1.2  samplex


위에서 봤듯 `build`는 *템플릿*과 *samplex*라는 두 객체를 돌려줍니다. samplex는 실행 시점의 레시피입니다. 입력과 출력으로 이루어진 [유향 비순환 그래프](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.dagcircuit.DAGCircuit)(DAG)로, 랜덤화마다 템플릿 회로에 넣을 파라미터 값과 측정된 비트열에 적용할 후처리를 결정합니다.

In [ ]:
print(samplex)
fig = samplex.draw()
fig.show(renderer="notebook")
print(type(fig))

위의 출력 텍스트는 samplex의 입력과 출력을 보여 주고, DAG 다이어그램은 이를 그래프로 그려 줍니다. **Inputs** 목록은 비어 있습니다. `Twirl` 어노테이션만 있는 samplex는 실행 시점에 넘겨받을 데이터가 없기 때문입니다. `InjectNoise` 어노테이션이 더해지면 달라지는데, 2.4절에서 보게 됩니다. 두 개의 **Outputs**인 `parameter_values`와 `measurement_flips.meas`는 다이어그램에서 컬렉터 노드(나비넥타이 모양)입니다.

DAG 다이어그램은 양자 회로가 **아닙니다**. samplex의 고전적 데이터 흐름 그래프입니다. 회로는 (2.1.1절에서 그린) 템플릿이고, samplex는 랜덤화마다 그 파라미터 자리를 무엇으로 채울지 계산합니다. 그러니까 samplex 다이어그램의 노드는 계산 단계이고, 간선은 단계 사이를 오가는 고전 데이터 레지스터입니다.

samplex에는 세 종류의 노드가 나타납니다:
- **빨간 별**은 *샘플링* 노드입니다. `Twirl` 박스마다 하나씩 있고, 그 박스의 무작위 파울리를 뽑습니다. 우리의 토이 예제에는 박스가 둘(CZ 박스와 측정 박스) 있습니다.
- **초록 원**은 처리 단계입니다. 파울리를 CZ 너머로 전파하는 것(1.2절의 클리포드 켤레 연산이 그래프 노드로 그려진 것)과 레지스터를 자르거나 합치는 일을 합니다.
- **나비넥타이**는 데이터를 출력으로 모으는 *컬렉터*입니다. **파란색**은 `parameter_values`(`[num_randomizations, 12]`, 템플릿의 자유 파라미터)로, **보라색**은 `measurement_flips.meas`(`[num_randomizations, 1, 2]`, 측정 트월링을 되돌리는 비트 반전)로 모읍니다.

다음은 샘플링 단계입니다. `samplex.sample(...)` 메서드에 랜덤화 개수 `num_randomizations`를 지정해서 호출합니다. `samplex.sample(...)`을 호출할 때마다 트월 분포에서 서로 독립인 무작위 파울리 집합 `num_randomizations`개를 뽑습니다. 결과는 회로가 필요로 하는 형태로 이미 변환되어 돌아옵니다. 템플릿을 채울 `parameter_values`와, 판독을 사후에 보정할 `measurement_flips`입니다.

In [ ]:
outputs = samplex.sample({}, num_randomizations=5)

print("parameter_values.shape     :", outputs["parameter_values"].shape)
print("measurement_flips.meas.shape:", outputs["measurement_flips.meas"].shape)
print()
print("First two parameter draws:")
print(outputs["parameter_values"][:2])
print()
print("Bit-flip corrections:")
print(outputs["measurement_flips.meas"][:, 0, :])

여기서는 랜덤화를 5개 샘플링했습니다. 출력은 `parameter_values`와 `measurement_flips`라는 두 키를 가진 딕셔너리입니다. 토이 모델의 템플릿에는 파라미터가 12개, 측정 트월에는 비트가 2개 있으니 출력의 모양은 예상과 일치합니다. 템플릿용 파라미터 12개짜리 세트 5개, 측정 트월용 비트 반전 보정 2개짜리 세트 5개를 얻었습니다.

이것은 순수하게 고전적인 단계입니다. 아직 아무 회로도 실행되지 않았습니다. 템플릿은 고정되어 있고, 랜덤화가 바뀔 때마다 달라지는 것은 파라미터 값과 측정 반전뿐입니다. 값이 채워진 회로들을 하드웨어에서 실행하는 것은 `Executor`의 일이고, 2.5절에서 다룹니다.

자연스럽게 두 가지 질문이 이어집니다. 랜덤화를 몇 개나 뽑아야 하는가, 그리고 `with circuit.box(...)` 블록을 일일이 손으로 쓰지 않고 회로를 박스로 묶으려면 어떻게 해야 하는가. 다음 절이 두 질문 모두에 답합니다.

## 2.2  랜덤화와 박싱 패스 매니저

첫 번째 질문은 랜덤화를 몇 개나 뽑을 것인가였습니다. 이를 제어하는 파라미터는 둘인데 헷갈리기 쉽습니다. `num_randomizations`(트월 분포에서 뽑는 독립적인 무작위 회로의 수)와 `shots_per_randomization`(각 회로에서 취하는 샷 수)입니다. 둘은 서로 트레이드오프 관계에 있습니다. 어느 비율을 넘어서면 샷당 통계 노이즈보다 트월링에서 오는 분산이 더 커지므로, 그때부터는 샷을 늘리는 것보다 랜덤화를 늘리는 쪽이 더 도움이 됩니다.

두 번째 질문은 `with circuit.box(...)` 블록을 손으로 쓰지 않고 회로를 박스로 묶는 방법이었습니다. [*박싱 패스 매니저*(boxing pass manager)](https://qiskit.github.io/samplomatic/guides/transpiler.html)가 그 답입니다. `generate_boxing_pass_manager(...)`는 회로를 자동으로 박스로 묶고 어노테이션까지 붙여 주는 트랜스파일러 패스입니다. 이 절에서는 그 옵션 중 세 가지, `enable_gates`, `enable_measures`, `twirling_strategy`를 사용합니다. `inject_noise_*` 옵션은 2.3절에서, 측정 어노테이션은 2.4절에서 다룹니다.

2.1절에서 쓴 토이 회로(CZ 뒤에 측정)를 그대로 다시 박스로 묶어 봅시다. 이번에는 손으로 박스 블록을 쓰는 대신 패스 매니저로 자동으로 묶습니다.

In [ ]:
# create the raw toy circuit
raw_toy = QuantumCircuit(2, 2)
raw_toy.cz(0, 1)
raw_toy.measure([0, 1], [0, 1])

# transpile the toy circuit using isa_pm we defined at the start of the lab
raw_toy_isa = isa_pm.run(raw_toy)

# create the boxing pass manager
twirl_only_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    twirling_strategy="active",
)

# box the toy circuit using the boxing pass manager
boxed_toy = twirl_only_pm.run(raw_toy_isa)
boxed_toy.draw("mpl", idle_wires=False)

박싱 패스 매니저는 박스가 없는 맨 회로를 받아서, 2.1절에서 손으로 만들었던 것과 같은 박스 회로를 자동으로 만들어 냅니다.

- `enable_gates=True`는 2큐비트 게이트를 `Twirl` 박스로 감쌉니다
- `enable_measures=True`는 측정을 박스로 감쌉니다
- `twirling_strategy="active"`는 각 박스가 실제로 작용하는 큐비트만 트월링합니다

회로 다이어그램에는 이렇게 생긴 두 박스가 빨간색으로 표시되고, 각각에 기본 `Twirl` 어노테이션이 붙어 있습니다(다이어그램에는 보이지 않습니다).

층이 많은 회로라면 박싱 패스 매니저를 쓰는 것이 현실적인 방법입니다. 층마다 `with circuit.box(...)`를 손으로 쓰는 방식은 규모가 커지면 직접 작성하는 것이 무리이기 때문입니다.

## 2.3  `InjectNoise` 어노테이션과 `NoiseLearnerV3`

2.2절에서 **박싱 패스 매니저**로 회로를 자동으로 박스로 묶을 수 있다는 것을 봤습니다. 지금까지 본 박스에는 `Twirl` 어노테이션만 붙어 있었습니다. 파울리 트월링은 각 층의 결맞음 오류를 *확률적 파울리 채널*로 바꿔 주어 노이즈를 다루기 쉽게 만들어 준다는 점에서 강력합니다. 희소한 파울리 채널은 임의의 노이즈 과정보다 훨씬 단순한 대상이니까요. 하지만 트월링만으로는 노이즈가 없어지지 않습니다.

이 절에서는 `InjectNoise` 어노테이션을 소개합니다. 이 어노테이션을 `NoiseLearnerV3`와 함께 사용해서 각 박스의 노이즈를 학습할 것입니다. 이렇게 해 두면 3장에서 **PNA**나 **SLC** 같은 고급 오류 완화 기법을 구현할 수 있게 됩니다.

이 곳에서 배울 내용은 세 단계로 진행됩니다. 먼저 2.3.1절에서 박싱 패스 매니저를 확장해 모든 게이트 박스에 `InjectNoise` 어노테이션 — `ref` 문자열을 키로 하는, 선언된 슬롯 — 이 붙게 합니다. 다음 2.3.2절에서 `NoiseLearnerV3`를 백엔드에서 실행해 각 슬롯을 학습된 파울리-린드블라드 모델로 채웁니다. 마지막으로 2.3.3절에서 돌려받은 노이즈 모델을 들여다보며 어떤 오류 생성원이 층을 지배하는지 확인합니다.

### 2.3.1  슬롯 선언하기: 게이트 박스마다 `InjectNoise`

아래 셀에서는 먼저 `noise_learning_boxing_pm`이라는 이름의 새 박싱 패스 매니저를 만듭니다. 위의 `twirl_only_pm`에서 쓴 옵션은 유지하고, 모든 게이트 박스가 `Twirl`과 함께 `InjectNoise` 어노테이션을 갖도록 `inject_noise_*` 옵션 두 개를 새로 추가합니다. 그런 다음 이 박싱 패스 매니저를 토이 회로에 적용합니다.

이어서 `find_unique_box_instructions`가 회로에서 *서로 다른* 게이트 층들을 추출합니다. `NoiseLearnerV3`가 실제로 특성화할 층이 바로 이것입니다. (단일 큐비트 드레싱까지 무시하면) 동등한 박스들은 같은 노이즈 모델을 공유하기 때문입니다. 동등한 박스가 많은 회로에서는 특히 효율적입니다. 노이즈 학습기가 *고유한* 층의 노이즈만 학습하면 되니까요.

In [ ]:
noise_learning_boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="no_modification",
)

boxed_toy = noise_learning_boxing_pm.run(raw_toy_isa)

unique_toy_layers = find_unique_box_instructions(
    boxed_toy,
    normalize_annotations=None,
    undress_boxes=True,
)


이전과 마찬가지로 이 박싱 패스 매니저는 박스 두 개를 만듭니다. CZ 층용 하나, 측정용 하나입니다. 이번에는 CZ 게이트 박스에 (`inject_noise_targets="gates"`에서 온) `InjectNoise`와 `Twirl` 어노테이션이 함께 붙고, 측정 박스는 `Twirl`만 유지합니다.

`inject_noise_strategy="no_modification"` 옵션은 (`find_unique_box_instructions` 기준으로) 동등한 박스 모두에 *같은* `ref`의 InjectNoise 어노테이션을 배정한다는 뜻입니다. `ref`는 노이즈를 주입할 파울리-린드블라드 맵을 유일하게 식별하는 문자열입니다. 똑같이 생긴 박스들에 같은 라벨을 붙여 준다고 생각하면 됩니다.

3장에서는 이 옵션을 `inject_noise_strategy=uniform_modification`(PNA)과 `inject_noise_strategy=individual_modification`(SLC)으로 바꿔야 합니다.

박싱 패스 매니저에 대한 더 자세한 내용은 [여기](https://qiskit.github.io/samplomatic/api/auto/samplomatic.transpiler.generate_boxing_pass_manager.html#samplomatic.transpiler.generate_boxing_pass_manager) 문서를 참고하세요.

In [ ]:
for i, inst in enumerate(unique_toy_layers):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a else "(none)"
    print(f"Layer {i}: ref = {ref}")
    display(inst.operation.body.draw("mpl", fold=-1, idle_wires=False))

위의 셀에서는 고유한 층마다 `ref` 문자열을 출력하고 해당 층의 회로 다이어그램을 그렸습니다.

`InjectNoise` 어노테이션이 붙은 고유 층은 각자 자기만의 `ref` 문자열을 가집니다. 출력을 보면 CZ 박스에는 `ref`가 있지만, `InjectNoise` 어노테이션이 없는 측정 층에는 없습니다.

`ref`는 `NoiseLearnerV3`가 각 고유 박스의 노이즈를 학습한 뒤 보고할 때 쓰는 라벨입니다. 같은 `ref` 라벨을 실행 시점에 `InjectNoise`가 다시 사용해서, 학습된 노이즈 모델을 찾아옵니다.

이제 슬롯은 선언됐지만 비어 있습니다. `NoiseLearnerV3`가 그 안에 *무엇을* 써넣는지는 다음 소절에서 정리합니다.

#### `NoiseLearnerV3`가 학습하는 것

`NoiseLearnerV3`는 각 층을 희소한 *파울리-린드블라드* 채널

$$\Lambda \;=\; \exp(\mathcal{L}), \qquad \mathcal{L}(\rho) \;=\; \sum_k \lambda_k\,(P_k\,\rho\,P_k - \rho),$$

로 특성화합니다. van den Berg, Minev, Kandala & Temme가 [*Probabilistic error cancellation with sparse Pauli–Lindblad models on noisy quantum processors*, Nat. Phys. **19**, 1116–1121 (2023)](https://www.nature.com/articles/s41567-023-02042-2)에서 정식화한 모델을 따릅니다. 여기서 $\mathcal{L}$은 초연산자(superoperator, 밀도 행렬 $\rho$에 작용합니다)이고, $\Lambda = e^{\mathcal{L}}$은 그 형식적 지수입니다. 생성원 $\{P_k\}$가 에르미트 파울리($P_k^\dagger = P_k$, $P_k^2 = I$)이기 때문에 소산 항은 오른쪽과 같은 형태로 줄어듭니다.

생성원(generator)의 집합은 층의 연결성이 정합니다. 관여하는 큐비트마다 1-체 파울리 $X_i, Y_i, Z_i$ 세 개가, 게이트 간선마다 항등이 아닌 2-체 파울리 $\{XX, XY, \ldots, ZZ\}$ 아홉 개가 들어갑니다. 예를 들어 큐비트 $(0,1)$의 CZ 하나짜리 층이라면 모델의 생성원은 $2\times 3 + 9 = 15$개입니다. 층 노이즈의 구조를 정의하는 것이 바로 이 생성원들의 레이트(rate)이고, `NoiseLearnerV3`가 학습하는 값이 그 레이트입니다. 2.3.2절에서 토이 회로의 레이트를 학습하고, 2.3.3절에서 이를 들여다보며 노이즈의 구조를 살펴보겠습니다.

*참고*: 이 모델이 담지 못하는 것도 있습니다:
- **결맞음 오류**: 결맞음 오류는 이미 트월링으로 확률적 파울리 노이즈가 되었다고 가정합니다
- **층 간 크로스토크**(crosstalk): 층의 연결성 밖에 있는 생성원은 나타나지 않습니다. 예컨대 생성원 $X_0 X_2$는 큐비트 $q_0$과 $q_2$에 작용하는 게이트가 없는 층에는 나타날 수 없습니다.

노이즈 학습기는 정확도와 QPU 시간을 맞바꾸는 세 옵션으로 설정합니다. `num_randomizations`(구성마다 독립적인 무작위 벤치마킹 회로 수), `shots_per_randomization`(회로당 샷 수), `layer_pair_depths`(층을 반복하는 깊이 $d$의 목록. 학습기는 이렇게 얻은 충실도-깊이 감쇠 곡선을 피팅해서 각 생성원의 레이트를 추출합니다)입니다.

### 2.3.2  토이 회로로 노이즈 학습 작업 실행하기

층을 박스로 묶고 슬롯 선언까지 마쳤으니, 이제 백엔드로 보냅니다. 첫 토이 회로 예제이므로 QPU 시간이 짧게 끝나도록 설정을 작게 잡습니다(`num_randomizations=5`, `shots_per_randomization=20`, `layer_pair_depths=[1, 2]`).

*참고:*
다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `TOY_NOISE_LEARN_JOB_ID` 파라미터에 붙여넣고 `SUBMIT_TOY_NOISE_JOB = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 3초입니다(ibm_pittsburgh 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
TOY_NOISE_LEARN_JOB_ID = None   # paste job_id here on re-run
SUBMIT_TOY_NOISE_JOB   = False   # set True to submit a fresh learning job

nl_options = NoiseLearnerV3Options(
    num_randomizations=5,
    shots_per_randomization=20,
    layer_pair_depths=[1, 2],
)
learner = NoiseLearnerV3(backend, nl_options)
learner.options.environment.job_tags = ["qgss26"]

if TOY_NOISE_LEARN_JOB_ID is not None:
    learner_job = service.job(TOY_NOISE_LEARN_JOB_ID)
    print(f"Re-using saved job: {TOY_NOISE_LEARN_JOB_ID}")
elif SUBMIT_TOY_NOISE_JOB:
    learner_job = learner.run(unique_toy_layers)
    TOY_NOISE_LEARN_JOB_ID = learner_job.job_id()
    print(f"Submitted: {TOY_NOISE_LEARN_JOB_ID}")
else:
    print("Set SUBMIT_TOY_NOISE_JOB=True to submit a fresh job, "
          "or paste a saved job id into TOY_NOISE_LEARN_JOB_ID and re-run.")



노이즈 학습 작업을 제출했으면, 아래 셀을 실행해 작업 상태를 확인하세요. 상태는 `QUEUED`, `RUNNING`, `DONE` 중 하나일 것입니다. 작업이 끝나면 다음 셀로 넘어가세요.

In [ ]:
learner_job = service.job(TOY_NOISE_LEARN_JOB_ID)
print(f"{TOY_NOISE_LEARN_JOB_ID}  (status: {learner_job.status()})")

In [ ]:
if learner_job.status() == "DONE":
    toy_noise_result = learner_job.result()
else:
    print(f"Not done yet (status={learner_job.status()}). Re-run cell when DONE.")

In [ ]:
if 'toy_noise_result' in dir() and toy_noise_result is not None:
    toy_refs_to_noise = toy_noise_result.to_dict(unique_toy_layers, require_refs=False)
    print(f"toy_refs_to_noise has {len(toy_refs_to_noise)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


위의 셀에서는 `to_dict` 메서드로 노이즈 학습 작업의 결과를 `toy_refs_to_noise`에 모았습니다. 학습된 층마다 파울리-린드블라드 모델이 하나씩 있어야 하고, 각각 박스 회로에서 쓴 것과 같은 `ref`로 라벨이 붙어 있어야 합니다. 토이 회로에서 `InjectNoise` 어노테이션이 붙은 층은 하나(CZ 층)뿐이라는 것을 알고 있으니, 노이즈 학습 작업도 학습된 노이즈 모델을 하나만 돌려줘야 합니다. 실제로 그렇다는 것을 확인할 수 있습니다.

이제 슬롯이 채워졌습니다. 실제로 무엇이 학습됐는지, 2.3.3절에서 CZ 층 하나의 학습된 노이즈 모델을 들여다봅니다.

### 2.3.3  학습된 노이즈 모델 들여다보기

아래 셀에서는 `PauliLindbladMap.to_sparse_list()` 메서드로 노이즈 모델을 `(pauli, qubits, rate)` 튜플의 나열로 돌려받습니다. $\mathcal{L}$의 0이 아닌 생성원마다 튜플이 하나씩 있습니다.

이 튜플들을 레이트 순으로 정렬해서 CZ 층을 지배하는 오류 채널이 무엇인지 보고, 상위 10개 생성원을 출력합니다. 층 단위 오류 완화 전략은 전부 이 정보를 보고 어디에 예산을 쓸지 정합니다. PNA, PEC, SLC에서 이것이 어떻게 구현되는지는 3장에서 보게 됩니다.

In [ ]:
if "toy_refs_to_noise" not in globals() or not toy_refs_to_noise:
    print("No learned toy noise model yet. Run the noise-learning result cell above first.")
else:
    ref0, plm0 = next(iter(toy_refs_to_noise.items()))
    generators = plm0.to_sparse_list()
    generators.sort(key=lambda g: -abs(g[2]))

    print(f"Layer ref = {ref0}, total generators = {len(generators)}\n")
    print(f"  {'Pauli':<6}{'qubits':<14}{'rate':>10}")
    print(f"  {'-'*6}{'-'*14}{'-'*10}")
    for pauli, qubits, rate in generators[:10]:
        print(f"  {pauli:<6}{str(qubits):<14}{rate:>10.4e}")


아직 소개할 어노테이션이 하나 남았습니다. $Z$가 아닌 기저에서의 측정을 위한 `ChangeBasis`입니다. 바로 다음 2.4절에서 다룹니다.

## 2.4  `ChangeBasis` 어노테이션

`ChangeBasis`는 $Z$가 아닌 기저에서 측정할 때 필요합니다. `InjectNoise`와 같은 패턴을 따릅니다. 어노테이션은 정적으로 붙여 두고, 값은 실행 시점에 공급하는 방식입니다. 측정 박스에 붙이며, 값은 samplex의 `basis_changes` 입력을 통해 실행 시점에 바인딩됩니다.

2장의 나머지에서는 `ChangeBasis`를 쓰지 않습니다. 2.6절에서 다룰 관측량이 계산 기저에서 대각이기 때문입니다. 하지만 3장에서 PNA가 $Z$ 아닌 항을 포함한 노이즈 완화 관측량 $\tilde{O}$를 만들어 내면 그때 사용하게 됩니다.

아래 셀은 `measure_annotations="all"`이 측정 박스에 `Twirl`과 `ChangeBasis`를 모두 붙이고, 그 결과 `samplex.inputs()`에 `basis_changes.<ref>` 슬롯이 추가되는 것을 보여 줍니다.

In [ ]:
# A boxing pass manager that ALSO annotates the measurement box with
# ChangeBasis. 
change_basis_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    measure_annotations="all",
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="no_modification",
)

demo_boxed = change_basis_pm.run(raw_toy_isa)
_, demo_samplex = build(demo_boxed)
print(demo_samplex)

이제 samplex가 **Inputs**를 두 개 갖습니다. 박스에 `Twirl` 어노테이션만 있던 2.1.2절에서는 하나도 없었는데 말이죠. 두 입력은 다음과 같습니다:
- `basis_changes.basisX`(X는 정수): 측정 시점에 적용할, 랜덤화마다 달라지는 기저 회전입니다. 심플렉틱 방식으로 인코딩됩니다 — `I=0, Z=1, X=2, Y=3`
- `pauli_lindblad_maps.<ref>`: 게이트 박스의 학습된 노이즈 모델

`ChangeBasis`와 `InjectNoise`를 회로에 추가하자, samplex가 실행 시점에 받기를 기대하는 데이터가 달라졌습니다. (일반적으로 `basisX`, `rYYYY` 꼴인데 X는 정수, Y는 정수나 문자입니다) 이 `ref` 문자열들이 그 데이터를 공급할 때 쓸 딕셔너리 키가 됩니다.

**Outputs**에는 `pauli_signs`도 새로 생깁니다. `InjectNoise`가 있으면 랜덤화마다 노이즈 모델에서 파울리 오류를 하나 샘플링하는데, 음수 레이트 인자의 홀짝이 $\pm 1$ 보정이 되어 나중에 후처리가 기댓값에 적용합니다.

즉 여기서도 같은 패턴이 이어집니다. 어노테이션이 필요한 것(기저 변경, 노이즈 모델, 트월 등)을 *선언*하면, samplex가 그 슬롯을 *드러내고* 실행 시점에 바인딩된 결과를 보고합니다. 2.5절에서 이 슬롯들에 값을 공급하고 `Executor`로 프로그램을 실행합니다.

## 2.5  `Executor`로 작업 제출하기

회로의 노이즈를 학습했으니, 이제 [**Executor 프리미티브**](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-executor)로 작업을 제출합니다. `Executor`는 Samplomatic 어노테이션을 존중하는 Runtime 프리미티브입니다.

잠깐 복습하자면, **Sampler**와 **Estimator** 프리미티브는 [PUB(Primitive Unified Bloc)](https://quantum.cloud.ibm.com/docs/en/guides/primitive-input-output#pubs)을 입력으로 받습니다. PUB은 `QuantumCircuit`과 파라미터, 그리고 Estimator라면 관측량까지를 입력으로 담는 튜플입니다.

Executor 프리미티브의 입력과 출력은 Sampler·Estimator와는 상당히 다릅니다. PUB 목록 대신 `QuantumProgram`을 받는데, 그 안에는 `QuantumProgramItem` 객체들의 목록이 들어 있습니다. 이 컨테이너 클래스들은 단순한 튜플 자료 구조인 PUB보다 유연합니다. `QuantumProgramItem`은 다음 중 하나입니다:
- `CircuitItem`: 회로와 그 파라미터 값을 담습니다
- `SamplexItem`: 템플릿 회로, samplex, samplex 인자를 담습니다

여기서는 `SamplexItem`을 쓰는 Executor 작업에 집중합니다. 먼저 `build`로 토이 회로의 템플릿과 samplex를 빌드합니다:

In [ ]:
toy_template, toy_samplex = build(boxed_toy)

print(toy_samplex)

다음으로 samplex 인자를 구성합니다. `samplex_arguments`의 키는 `samplex.inputs()`가 돌려주는 키와 정확히 일치해야 합니다. `noise_scales.<ref>`나 `basis_changes.<ref>`처럼 자동 생성된 ref 이름까지 그대로여야 합니다.

In [ ]:
samplex_arguments = (
    toy_samplex.inputs()
    .make_broadcastable()
    .bind(pauli_lindblad_maps=toy_refs_to_noise)
)

이제 `QuantumProgram`을 구성할 수 있습니다. `append_samplex_item(...)` 메서드로 `QuantumProgram`에 `SamplexItem`을 추가합니다(여러 개가 공존할 수 있고, 완화된 작업을 배치로 묶을 때 이 방식을 씁니다).

In [ ]:
program = QuantumProgram(shots=64)
program.append_samplex_item(
    toy_template,
    samplex=toy_samplex,
    samplex_arguments=samplex_arguments,
    shape=(16,),
)

이제 QPU에서 Executor 작업을 실행할 준비가 끝났습니다.

*참고:* 다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `TOY_EXECUTOR_JOB_ID` 파라미터에 붙여넣고 `SUBMIT_TOY_EXEC_JOB = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 2초입니다(ibm_fez 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
TOY_EXECUTOR_JOB_ID = None # paste job_id here on re-run
SUBMIT_TOY_EXEC_JOB = False # set True to submit a fresh executor job

executor = Executor(backend)
executor.options.environment.job_tags = ["qgss26"]

if TOY_EXECUTOR_JOB_ID is not None:
    toy_job = service.job(TOY_EXECUTOR_JOB_ID)
    print(f"Re-using saved job: {TOY_EXECUTOR_JOB_ID}")
elif SUBMIT_TOY_EXEC_JOB:
    toy_job = executor.run(program)
    TOY_EXECUTOR_JOB_ID = toy_job.job_id()
    print(f"Submitted Executor job: {TOY_EXECUTOR_JOB_ID}")
else:
    print("Set SUBMIT_TOY_EXEC_JOB=True to submit a fresh executor job, "
          "or paste a saved job id into TOY_EXECUTOR_JOB_ID and re-run.")


아래 셀로 작업 상태를 확인하세요:

In [ ]:
toy_job = service.job(TOY_EXECUTOR_JOB_ID)
print(f"{TOY_EXECUTOR_JOB_ID}  (status: {toy_job.status()})")

결과를 꺼내서 큐비트 0과 큐비트 1 각각의 $Z$ 기댓값을 따로 살펴봅니다:

In [ ]:
if toy_job.status() == "DONE":
    toy_result = toy_job.result()
    data    = toy_result[0]
    meas    = data["c"]
    flips   = data["measurement_flips.c"]

    # Shape sanity check: layouts can vary across SDK versions.
    print(f"meas shape  : {meas.shape}")
    print(f"flips shape : {flips.shape}")

    corrected = np.bitwise_xor(meas, flips)
    z_vals    = 1 - 2 * corrected

    # Average over every leading axis (randomizations, shots, ...) so only the
    # final qubit axis remains. This is robust to the (randomizations, shots,
    # bits) vs (shots, bits) shape difference.
    z_means = z_vals.reshape(-1, z_vals.shape[-1]).mean(axis=0)

    print(f"\n<Z_0> = {z_means[0]:+.4f}")
    print(f"<Z_1> = {z_means[1]:+.4f}")
else:
    print(f"Not done yet (status={toy_job.status()}). Re-run cell when DONE.")


이상적인 값은 두 큐비트 모두 $\langle Z_i \rangle = +1$입니다. CZ는 $|00\rangle$에 아무 일도 하지 않으므로, 최종 상태가 $|00\rangle$ 그대로여야 하기 때문입니다. 이 상태에서 벗어난 만큼이 하드웨어 노이즈입니다. 보통의 실행에서는 편차가 퍼센트 수준이고, 그 크기는 백엔드, 큐비트 배치, 대기열에 들어간 시점의 캘리브레이션, 샷 예산에 따라 달라집니다.

기여하는 요인은 대체로 세 가지입니다. 두 물리 큐비트의 **판독 오류**, 단일 큐비트 드레싱 회전이 도는 동안의 **유휴 결어긋남**(decoherence), 그리고 CZ에서 남은 **잔여 확률적 파울리 오류**입니다.

편차가 두 큐비트 사이에 비대칭인 것도 흔한 일인데, `optimization_level=0`으로 트랜스파일했기 때문입니다. 큐비트 쌍이 충실도가 아니라 인덱스 순서로 골라졌으니 두 큐비트의 오류 예산이 서로 다릅니다. 2.3.3절에서 본 지배적 생성원 레이트($10^{-3}$ 정도)는 자릿수 감각을 주는 진단 지표이지만, 관측된 편차에는 판독 오류, 유휴 시간, 배치에 따른 캘리브레이션, 누적된 박스 오버헤드까지 함께 들어가므로 레이트만으로 편차가 정해지지는 않습니다.

2.3.3절에서 본 것처럼 작은 층별 레이트는, 완화 효과가 눈에 보이려면 여러 층에 걸쳐 쌓여야 합니다. 2.6절에서는 같은 파이프라인을 더 깊은 회로(1차원 이징 미러)에 적용하고, 3장의 PNA와 SLC가 소비할 `refs_to_noise_models` 딕셔너리를 만들어 냅니다.


## 2.6  중간 연습: 1차원 이징 사슬

지금까지는 2큐비트 토이 CZ 회로로 Samplomatic의 도구를 하나씩 소개하고 전체 워크플로를 익혔습니다. 여기서부터는 1차원 횡자기장 이징 사슬(transverse-field Ising chain)을 다룹니다. 3장에서도 계속 사용할 시스템입니다.

박싱 → 빌드 → 노이즈 학습 → 실행이라는 워크플로는 그대로이고, 이제는 회로가 충분히 깊어서 층별 레이트가 측정 가능한 편차로 쌓입니다.

이 절의 끝에는 2장에서 다룬 내용 전부를 시험해 보는 연습 문제가 두 개 있습니다. Exercise 2에서는 `NoiseLearnerV3`가 특성화할 층이 많아지도록 더 깊은 이징 미러를 박스로 묶고, Exercise 3에서는 그 결과물(박스 회로, 학습된 노이즈, samplex, `QuantumProgram`)을 `Executor`가 소비하는 형태로 조립합니다.

### 2.6.1  해밀토니안과 트로터 회로

다룰 시스템은 **1차원 횡자기장 이징 사슬**이고, 해밀토니안은 다음과 같습니다:

$$H \;=\; -J \sum_{\langle i,j\rangle} Z_i Z_j \;+\; h \sum_i X_i.$$

이 해밀토니안은 유틸리티 규모의 표준 벤치마크입니다. (2차원 heavy-hex 격자 위의) 같은 해밀토니안이 IBM의 127큐비트 유틸리티 시연을 이끌었습니다([Kim et al., *Evidence for the utility of quantum computing before fault tolerance*, Nature **618**, 500–505 (2023)](https://www.nature.com/articles/s41586-023-06096-3)). 여기서 1차원 사슬을 쓰는 이유는 벽돌쌓기(brickwork) 구조를 살펴보기 쉽기 때문이고, 3장의 PNA·SLC 튜토리얼도 같은 시스템으로 이어집니다.

아래에서 도우미 함수 `construct_ising_circuit(num_qubits, num_trotter_steps, rx_angle, barrier=True)`를 정의합니다. 이 함수는 두 큐비트에 $S^\dagger$를 적용한 뒤 $\mathrm{CZ}$를 거는 방식으로 각 이징 회로를 만듭니다. 세 게이트 모두 대각이고 서로 교환하므로, 이 나열은 전역 위상을 제외하면 $R_{ZZ}(-\pi/2)$와 같습니다. 파라미터화된 `RZZ` 대신 이렇게 써 두면 모든 2큐비트 게이트가 CZ로 유지되어, 2.1–2.5절의 박싱·드레싱 장치를 그대로 적용할 수 있습니다.

이 회로는 고정 각도의 킥트-이징(kicked-Ising) 계열 트로터 스텝입니다. 횡자기장 회전은 `rx_angle`로 제어하고, ZZ 상호작용은 고정된 CZ 기반 블록으로 구현합니다. 이징 회로의 물리를 여기서 더 깊이 파고들지는 않겠습니다. 이 랩의 초점은 노이즈 학습과 오류 완화이고, 이징 회로는 다루기 좋은 예제를 제공해 줄 뿐입니다.

In [ ]:
def construct_ising_circuit(num_qubits: int,
                            num_trotter_steps: int,
                            rx_angle: float,
                            barrier: bool = True) -> QuantumCircuit:
    qc = QuantumCircuit(num_qubits)
    for _ in range(num_trotter_steps):
        qc.rx(rx_angle, range(num_qubits))
        if barrier:
            qc.barrier()
        for first_qubit in (1, 2):
            for idx in range(first_qubit, num_qubits, 2):
                qc.sdg([idx - 1, idx])
                qc.cz(idx - 1, idx)
        if barrier:
            qc.barrier()
    return qc

ising = construct_ising_circuit(num_qubits=4, num_trotter_steps=1, rx_angle=np.pi / 8)
ising.draw("mpl", fold=-1)

### 2.6.2  미러 트릭

3장에서는 1차원 이징 사슬 회로에 대해 *완화하지 않은* 결과와 *오류 완화한* 결과를 비교할 것입니다. 구현한 오류 완화 기법의 효과를 검증하려면 우리가 목표로 삼을 이상적인 결과를 알아야 합니다. 임의의 회로라면 별도의 고전 시뮬레이션이 필요할 텐데, 유틸리티 규모에서는 그 시뮬레이션 자체가 비싸거나 아예 불가능합니다. 이때 등장하는 것이 미러 트릭(mirror trick)입니다.

미러 트릭은 관심 있는 회로의 끝에 그 역회로를 이어 붙입니다. 즉 회로를 앞으로 한 번, 뒤로 한 번 적용하는 것입니다. 노이즈가 없는 이상적인 양자 컴퓨터라면 양자 상태는 초기 상태 $|0\rangle^{\otimes N}$로 되돌아옵니다. 여기서 나오는 유용한 결론은, 값비싼 고전 시뮬레이션 없이도 임의의 관측량에 대한 이상적인 기댓값을 쉽게 계산할 수 있다는 것입니다. 예를 들어 모든 큐비트에서 $Z$의 이상적인 기댓값이 1, 즉 모든 $i$에 대해 $\langle Z_i \rangle = +1$이라는 것을 압니다. 측정된 $\langle Z_i \rangle$가 $+1$에서 벗어난 만큼이 하드웨어 노이즈 때문입니다. 미러 회로가 오류 완화의 표준 벤치마크인 이유입니다.

Qiskit에서는 `ising.compose(ising.inverse())`로 구현합니다. $U$ 뒤에 $U^\dagger$를 이어 붙이는 것으로, Qiskit의 오른쪽에서 왼쪽 관례로는 $U_\text{mirror} = U^\dagger U = I$가 됩니다.

In [ ]:
mirror = ising.compose(ising.inverse())
mirror.measure_all()
mirror_isa = isa_pm.run(mirror)
mirror_isa.draw("mpl", fold=-1, idle_wires=False, scale=0.5)

### 2.6.3  이징 미러 박싱하기

회로도 있고 워크플로도 있으니, 다음 단계는 둘을 합치는 것입니다.

아래에서 미러 회로를 `NoiseLearnerV3`가 소비하는 형태로 바꿉니다. 2.3.1절의 `noise_learning_boxing_pm`을 고치지 않고 재사용합니다. 두 회로 모두 CZ 얽힘 게이트로 만들어져 있어 같은 박싱 전략이 통하고, 각 게이트 층은 `Twirl` + `InjectNoise` 박스로 감싸집니다.

그다음 `find_unique_box_instructions(...)`가 박스 회로를 *구조적으로 서로 다른* 층들로 접어 줍니다. 동등한 박스들은 노이즈 모델을 공유하므로 `NoiseLearnerV3`는 고유한 층마다 한 번씩만 특성화하면 된다는 점을 기억하세요.
- `undress_boxes=True`는 고유 층을 찾기 위한 비교 전에 무작위 파울리 드레싱을 벗겨 냅니다. 드레싱은 박스마다 다르지만 그 밑의 층은 같기 때문입니다.
- `normalize_annotations=None`은 박스 어노테이션 중 `Twirl`과 `InjectNoise`만 남기고 나머지는 버립니다

In [ ]:
boxed_circuit_ising = noise_learning_boxing_pm.run(mirror_isa)
unique_layers_ising = find_unique_box_instructions(
    boxed_circuit_ising,
    normalize_annotations=None,
    undress_boxes=True,
)

print(f"Unique 2Q-gate layers: {len(unique_layers_ising)}")
for i, inst in enumerate(unique_layers_ising):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a is not None else "(none)"
    print(f"  layer {i}: ref = {ref}")

boxed_circuit_ising.draw('mpl', idle_wires=False)


고유 층을 추출했으니 들여다봅니다. (`NoiseLearnerV3`가 결과를 보고할 때 쓸) `ref` 문자열과 각 층의 게이트 구성을 확인합니다. 이징 회로에 박스는 5개 있지만 고유한 층은 3개뿐이고, 그중 하나는 측정 층이라는 것을 볼 수 있습니다.

In [ ]:

for i, inst in enumerate(unique_layers_ising):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a else "(none)"
    print(f"Layer {i}: ref = {ref}")
    display(inst.operation.body.draw("mpl", fold=-1, idle_wires=False))

### 2.6.4  노이즈 학습하기

박스 회로가 준비됐으니 이제 그 노이즈를 학습할 수 있습니다. `unique_layers_ising`을 `NoiseLearnerV3`에 넘기고, 결과를 각 층의 `InjectNoise.ref`를 학습된 `PauliLindbladMap`으로 잇는 딕셔너리 `refs_to_noise_models_ising`으로 조립합니다. 두 게이트 층의 특성화가 끝나면 두 층의 노이즈 프로파일이 얼마나 다른지 읽어낼 수 있습니다.

*참고:* 다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `NOISE_LEARN_JOB_ID_ISING` 파라미터에 붙여넣고 `SUBMIT_NOISE_JOB_ISING = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 10초입니다(ibm_fez 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
NOISE_LEARN_JOB_ID_ISING = None # paste job_id here on re-run
SUBMIT_NOISE_JOB_ISING   = False   # set True to submit a fresh learning job
learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_ISING is not None:
    learner_job_ising = service.job(NOISE_LEARN_JOB_ID_ISING)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_ISING}")
elif SUBMIT_NOISE_JOB_ISING:
    learner_job_ising = learner.run(unique_layers_ising)
    NOISE_LEARN_JOB_ID_ISING = learner_job_ising.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_ISING}")
else:
    print("Set SUBMIT_NOISE_JOB_ISING=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_ISING and re-run.")


In [ ]:
learner_job_ising = service.job(NOISE_LEARN_JOB_ID_ISING)
print(f"{NOISE_LEARN_JOB_ID_ISING}  (status: {learner_job_ising.status()})")

In [ ]:
if learner_job_ising.status() == "DONE":
    noise_learner_result_ising = learner_job_ising.result()
else:
    print(f"Not done yet (status={learner_job_ising.status()}). Re-run cell when DONE.")

In [ ]:
if 'noise_learner_result_ising' in dir() and noise_learner_result_ising is not None:
    refs_to_noise_models_ising = noise_learner_result_ising.to_dict(unique_layers_ising, require_refs=False)
    print(f"refs_to_noise_models_ising has {len(refs_to_noise_models_ising)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


딕셔너리가 준비됐습니다. 항목 하나가 학습된 층 하나입니다. 무엇이 학습됐는지 보기 위해, 아래 셀에서 두 층의 특성을 출력합니다. 층마다 생성원의 개수, 그중 레이트가 0이 아닌 것의 개수, 가장 큰 생성원 레이트, 층 전체 레이트의 합(총 노이즈), 그리고 가장 크게 기여하는 파울리 항을 출력합니다.

In [ ]:
if "refs_to_noise_models_ising" not in globals() or not refs_to_noise_models_ising:
    print("Fetch a completed Ising noise-learning result first.")
else:
    print("Layer comparison:\n")
    print(f"  {'ref':<10}{'#gens':>8}{'#nonzero':>10}{'max rate':>14}{'sum rates':>14}{'top generator':>22}")
    print("  " + "─" * 78)
    for ref, plm in refs_to_noise_models_ising.items():
        gens = plm.to_sparse_list()
        rates = [abs(r) for _, _, r in gens]
        nonzero = [r for r in rates if r > 0]
        top = max(gens, key=lambda g: abs(g[2]))
        top_label = f"{top[0]}@{tuple(top[1])} ({top[2]:.2e})"
        print(f"  {ref:<10}{len(gens):>8}{len(nonzero):>10}"
              f"{max(rates):>14.4e}{sum(rates):>14.4e}"
              f"{top_label:>22}")


두 고유 층은 벽돌쌓기 구조 안에서 비슷한 역할을 합니다. 하나는 짝수 결합(여기서는 CZ 게이트가 두 개인 층)을, 다른 하나는 홀수 결합(여기서는 CZ가 하나뿐인 층)을 구현합니다. 그런데 위의 표를 보면 학습된 노이즈는 서로 다릅니다.

`sum rates` 열은 각 층의 총 노이즈를, `top generator` 열은 어떤 채널이 지배적인지를 보여 주는데, 한 층의 지배적 채널이 다른 층에서는 약하거나 아예 없을 수 있습니다. `#nonzero` 열을 보면 한 층은 노이즈를 소수의 채널에 몰아넣고 다른 층은 더 많은 채널에 흩뿌린다는 것도 알 수 있습니다.

층마다 노이즈가 이렇게 다르다는 것이 오류 완화를 층 단위로 하고 싶은 이유입니다. 한 층의 지배적 생성원에 맞춘 일률적인 보정은 다른 층에서는 빗나가거나 과보정이 됩니다. Runtime의 회로 전체 `EstimatorOptions`(1.1절)가 메울 수 없는 간극이 정확히 이것입니다.

3장에서 PNA가 각 층 *자신의* 역채널을 노이즈 완화 관측량에 흡수하는 방법과, SLC가 각 층을 자기 라이트콘을 따라 스케일링하는 방법을 보게 됩니다. 두 방법 모두 방금 출력한 층별 레이트를 읽어서 작동합니다.

*참고:* 구체적인 결과는 사용하는 QPU 백엔드와 그로부터 학습된 노이즈에 따라 달라집니다. 노이즈는 드리프트하고 캘리브레이션은 매일 이루어집니다. 같은 QPU라도 노이즈는 시간이 지나면 달라집니다.

### Exercise 2 — 더 깊은 이징 미러 박싱하기

<div class="alert alert-block alert-success">

2.6.1–2.6.4절에서 4큐비트, 트로터 1스텝짜리 이징 회로로 노이즈 학습 워크플로 전체를 밟아 봤습니다.

이번에는 같은 워크플로를 더 깊은 회로에 직접 적용해 보세요. 여기서 만든 것을 Exercise 3에서 가져다가 `Executor` 프로그램으로 조립하게 됩니다.

- **회로:** 6큐비트 이징 사슬, 트로터 2스텝, `rx_angle = π/8`, `barrier=False`
- **미러:** `ising_ex2`를 compose하고, `barrier()`를 추가한 뒤, `ising_ex2.inverse()`를 compose하고, 마지막으로 `measure_all()`
- **트랜스파일:** `isa_pm`을 통해
- **박싱:** `noise_learning_boxing_pm`을 통해

앞쪽 절반과 역방향 절반 사이의 `barrier()`는 트랜스파일러의 최적화 패스가 미러를 항등 회로로 상쇄해 버리는 것을 막아 줍니다.

아래 셀을 채워서, `boxed_circuit_ex2`가 `build()`가 받아 주는 Twirl + InjectNoise 어노테이션이 붙은 박스 미러 회로가 되도록 만드세요.

</div>

In [ ]:
ising_ex2 = None
mirror_ex2 = None
boxed_circuit_ex2 = None

#TODO: Your code below.

# 1. Build the 6-qubit, 2-step Ising circuit (rx_angle = π/8, barrier=False)
ising_ex2 = ...

# 2. Build mirror_ex2: forward + barrier + inverse + measure_all
#    (The barrier keeps the transpiler from cancelling the mirror.)
mirror_ex2 = QuantumCircuit(num_qubits)
mirror_ex2.compose(..., inplace=True)   # forward
mirror_ex2.barrier()
mirror_ex2.compose(..., inplace=True)   # inverse
mirror_ex2.measure_all()

# 3. Transpile through isa_pm and box through noise_learning_boxing_pm
mirror_ex2_isa = ...
boxed_circuit_ex2 = ...

In [ ]:
# grade your answer
grade_lab3_ex2(ising_ex2, mirror_ex2, boxed_circuit_ex2)

Exercise 2에서 여러분은 모든 게이트 층에 `InjectNoise` 슬롯이 붙은 박스 이징 미러 회로 `boxed_circuit_ex2`를 만들었습니다. 이 박스 회로는 이제 특성화할 준비가 됐습니다.

Exercise 3에서 실행 가능한 프로그램으로 조립하기 전에, 그 슬롯들을 채워 둡니다. 2.6.4절과 같은 `NoiseLearnerV3` 워크플로를 더 깊은 회로에 적용하는 것입니다. 실행 방식도 완전히 같습니다. 작업을 제출하고, 상태를 확인하고, 결과를 가져옵니다. 학습된 노이즈로부터 Exercise 3에 필요한 `{ref: PauliLindbladMap}` 딕셔너리, `refs_to_noise_models_ex3`를 만듭니다.

#### 노이즈 학습하기 (Exercise 3에 필요합니다):

*참고:* 다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `NOISE_LEARN_JOB_ID_EX3` 파라미터에 붙여넣고 `SUBMIT_NOISE_JOB_EX3 = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 4초입니다(ibm_fez 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
unique_layers_ex3 = find_unique_box_instructions(
    boxed_circuit_ex2,
    normalize_annotations=None,
    undress_boxes=True,
)

NOISE_LEARN_JOB_ID_EX3 = None   # paste a saved job id to re-fetch
SUBMIT_NOISE_JOB_EX3   = False              # set True to submit a fresh job
learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_EX3 is not None:
    learner_job_ex3 = service.job(NOISE_LEARN_JOB_ID_EX3)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_EX3}")
elif SUBMIT_NOISE_JOB_EX3:
    learner_job_ex3 = learner.run(unique_layers_ex3)
    NOISE_LEARN_JOB_ID_EX3 = learner_job_ex3.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_EX3}")
else:
    print("Set SUBMIT_NOISE_JOB_EX3=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_EX3 and re-run.")

In [ ]:
learner_job_ex3 = service.job(NOISE_LEARN_JOB_ID_EX3)
status = learner_job_ex3.status()

print(f"job_id : {NOISE_LEARN_JOB_ID_EX3}")
print(f"status : {status}")

if status == "DONE":
    print("\n  Ready — proceed to Exercise 3.")

작업이 `DONE`이 되면 아래 셀을 실행해 결과를 가져와서 `refs_to_noise_models_ex3`, 즉 층마다 학습된 모델이 하나씩 들어 있는 `{ref: PauliLindbladMap}` 딕셔너리로 모으세요. Exercise 3에서 이 객체가 필요합니다.

In [ ]:
noise_result_ex3 = learner_job_ex3.result()
refs_to_noise_models_ex3 = noise_result_ex3.to_dict(
    unique_layers_ex3, require_refs=False
)
print(f"Learned noise for {len(refs_to_noise_models_ex3)} layers: "
      f"{list(refs_to_noise_models_ex3.keys())}")

학습된 노이즈 모델은 날것의 레이트 숫자보다 그림으로 보는 쪽이 이해하기 쉽습니다. 아래 플롯은 각 층에서 가장 큰 노이즈 생성원 15개를 보여 주는데, 1-체(단일 큐비트) 항과 2-체(두 큐비트) 항을 색으로 구분했습니다. 각 층의 노이즈 모델을 지배하는 생성원이 무엇인지 뚜렷하게 보입니다.

In [ ]:
# plot the top 15 noise generators per layer
fig, axes = plt.subplots(1, len(refs_to_noise_models_ex3), figsize=(12, 4))
if len(refs_to_noise_models_ex3) == 1: axes = [axes]

for ax, (ref, plm) in zip(axes, refs_to_noise_models_ex3.items()):
    gens = sorted(plm.to_sparse_list(), key=lambda g: -abs(g[2]))[:15]
    labels = [f"{p}@{tuple(q)}" for p, q, _ in gens]
    rates  = [r for _, _, r in gens]
    colors = ['#2c7fb8' if len(p) == 1 else '#d95f0e' for p, _, _ in gens]
    ax.barh(range(len(labels)), rates, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Rate")
    ax.set_title(f"Layer {ref}")
    ax.grid(axis="x", alpha=0.3)

fig.legend(handles=[Patch(facecolor="#2c7fb8", label="1-qubit"),
                    Patch(facecolor="#d95f0e", label="2-qubit")],
           loc="upper right", fontsize=9)
fig.suptitle("Top 15 noise generators per layer")
plt.tight_layout()
plt.show()

이제 노이즈 모델, 구체적으로는 딕셔너리 `noise_result_ex3`가 있으니 Executor 작업을 실행할 수 있습니다.

Exercise 3에서는 `noise_result_ex3`와 `build()` 함수를 이용해 Executor 작업을 준비합니다. Exercise 2에서 만든 박스 회로로 템플릿과 samplex를 만드는 것부터 시작합니다.

### Exercise 3 — `Executor` 프로그램 조립하기

<div class="alert alert-block alert-success">

위의 노이즈 학습 작업으로 `refs_to_noise_models_ex3`를 얻었습니다. `boxed_circuit_ex2`의 층마다 노이즈 모델이 하나씩 들어 있는 `{ref: PauliLindbladMap}` 딕셔너리입니다.

이 연습 문제에서는 그 노이즈 모델을 `Executor`가 실행하는 자료 구조에 끼워 넣습니다. 2장에서 소개한 세 가지 도구 — **Samplomatic**의 `build`, 방금 만든 **NoiseLearnerV3** 결과, 그리고 **Executor**의 `QuantumProgram` — 를 한 번에 모두 사용합니다.

프로그램은 네 단계로 만듭니다:

- **빌드:** `build(boxed_circuit_ex2)`를 호출해 `template_ex3`와 `samplex_ex3`를 얻습니다
- **확인:** `samplex_ex3.inputs()`가 samplex가 기대하는 `pauli_lindblad_maps.<ref>` 슬롯들을 알려 줍니다. 이 ref들은 `refs_to_noise_models_ex3`의 키와 일치해야 합니다
- **바인딩:** `samplex_ex3.inputs()`에서 `make_broadcastable()`을 호출한 뒤, 학습된 딕셔너리로 `bind(pauli_lindblad_maps=...)`를 호출합니다 → `samplex_args_ex3`
- **조립:** `QuantumProgram(shots=64)`을 만들고 템플릿, samplex, 인자를 `append_samplex_item(...)`으로 추가합니다 → `program_ex3`

채점기는 프로그램이 올바르게 조립됐는지만 확인하며, 하드웨어에서 실행하지는 않습니다. 이어지는 선택 셀에서 이 프로그램을 실제 백엔드에서 돌리면 어떤 기댓값이 나오는지 볼 수 있습니다.

</div>

In [ ]:
#TODO: Add Your code here and set the missing values.

template_ex3      = None
samplex_ex3       = None
samplex_args_ex3  = None
program_ex3       = None


# 1. Build the template and samplex from boxed_circuit_ex2
template_ex3, samplex_ex3 = ...

# 2. Inspect what the samplex expects (look at the pauli_lindblad_maps.<ref> slots)
print(samplex_ex3.inputs())

# 3. Bind the learned noise models into the samplex inputs.
#    Start from samplex_ex3.inputs(), make it broadcastable, then bind the
#    refs_to_noise_models_ex3 dict to the `pauli_lindblad_maps` argument.
samplex_args_ex3 = (
    samplex_ex3.inputs()
    .make_broadcastable()
    .bind(pauli_lindblad_maps=...)
)

# 4. Assemble the QuantumProgram with a single samplex item
program_ex3 = QuantumProgram(shots=64)
program_ex3.append_samplex_item(...)

In [ ]:
# grade your answer
grade_lab3_ex3(
    template_ex3,
    samplex_ex3,
    samplex_args_ex3,
    program_ex3,
    refs_to_noise_models_ex3,
)

#### (선택) 하드웨어에서 프로그램 실행하기

이 단계는 선택 사항이며, 3장으로 넘어가는 데 꼭 필요하지는 않습니다. `program_ex3`를 실제 하드웨어에서 돌리면 무엇이 나오는지 보여 주는 단계입니다. 토이 회로의 `Executor` 작업을 제출했던 2.5절과 같은 패턴을 따릅니다.

여기서의 회로는 이징 미러 회로이므로 이상적인 결과는 모든 큐비트에서 $\langle Z_i\rangle = +1$입니다. 여기서 벗어난 만큼이 박싱-학습 파이프라인이 아직 남겨 둔 노이즈입니다. 3장에서 오류 완화 기법을 구현해 이 결과를 개선할 것입니다.

*참고:* 다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `EXECUTOR_JOB_ID_EX3` 파라미터에 붙여넣고 `SUBMIT_EXECUTOR_JOB_EX3 = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 2초입니다(ibm_pittsburgh 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
executor = Executor(backend)

EXECUTOR_JOB_ID_EX3      = None      # paste an Executor job_id here on re-run
SUBMIT_EXECUTOR_JOB_EX3  = False     # set True to submit a fresh Executor job
executor.options.environment.job_tags = ["qgss26"]

if EXECUTOR_JOB_ID_EX3 is not None:
    exec_job_ex3 = service.job(EXECUTOR_JOB_ID_EX3)
    print(f"Re-using saved job: {EXECUTOR_JOB_ID_EX3}")
elif SUBMIT_EXECUTOR_JOB_EX3:
    exec_job_ex3 = executor.run(program_ex3)
    EXECUTOR_JOB_ID_EX3 = exec_job_ex3.job_id()
    print(f"Submitted Executor job: {EXECUTOR_JOB_ID_EX3}")
else:
    print("Set SUBMIT_EXECUTOR_JOB_EX3=True to submit a fresh Executor job, "
          "or paste a saved job id into EXECUTOR_JOB_ID_EX3 and re-run.")

In [ ]:
exec_job_ex3 = service.job(EXECUTOR_JOB_ID_EX3)
print(f"{EXECUTOR_JOB_ID_EX3}  (status: {exec_job_ex3.status()})")

결과를 플롯합니다:

In [ ]:
if exec_job_ex3.status() == "DONE":
    exec_result_ex3 = exec_job_ex3.result()
    data  = exec_result_ex3[0]
    meas  = data["meas"]
    flips = data["measurement_flips.meas"]

    corrected = np.bitwise_xor(meas, flips)
    z_vals    = 1 - 2 * corrected
    z_means   = z_vals.reshape(-1, z_vals.shape[-1]).mean(axis=0)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(z_means)), z_means, color="#2c7fb8")
    ax.axhline(1.0, color="gray", ls="--", lw=1, label="ideal ⟨Z⟩ = +1")
    ax.set_xlabel("Qubit")
    ax.set_ylabel("⟨Z⟩")
    ax.set_ylim(0, 1.05)
    ax.set_title("Mirror expectation values (boxed + learned, pre-mitigation)")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"mean ⟨Z⟩ = {z_means.mean():+.4f}  (ideal = +1)")
else:
    print(f"Not done yet (status={exec_job_ex3.status()}). Re-run cell when DONE.")

#### 2장을 마치며:

축하합니다! Lab 3의 2장을 끝냈습니다. 이 장에서는 3장의 고급 오류 완화 기법을 구현하는 데 필요한 도구를 전부 소개했습니다.

Samplomatic + 노이즈 학습 워크플로 전체를 처음에는 단순한 2큐비트 토이 모델로, 다음에는 좀 더 복잡한 4큐비트 이징 사슬로, 마지막에는 6큐비트 이징 사슬로 밟아 봤습니다. 회로가 바뀌어도 워크플로는 매번 같았습니다.

- **Exercise 2**에서는 6큐비트 이징 미러 회로를 박스로 묶었고, 노이즈 학습 단계가 그 층들을 특성화했습니다.
- **Exercise 3**에서는 박스 회로, 앞서 학습한 노이즈, samplex를 `Executor`가 실행하는 `QuantumProgram`으로 조립했습니다.

그 과정에서 층별 노이즈 구조도 들여다봤습니다. `InjectNoise` 어노테이션이 붙은 모든 층에 달리는 `ref` 문자열이, 학습된 노이즈 정보를 붙여 두는 라벨이라는 것을 배웠습니다.

이징 미러 회로에서는 층들이 회로의 물리에서 비슷한 역할을 하는데도 층마다 학습된 노이즈 모델이 다르다는 것(특히 지배적 생성원이 다르다는 것)을 확인했습니다. 오류 완화가 **층 단위로** 작동하기를 바라는 이유가 정확히 여기에 있습니다.

#### 아직 빠져 있는 것 — 3장이 존재하는 이유:

중요한 점은, 2장에서 본 파이프라인은 노이즈를 측정하고 실행에 쓸 수 있게 준비할 뿐 노이즈를 제거하지는 않는다는 것입니다. 지금까지 본 것처럼 트월링은 결맞음 오류를 확률적 파울리 채널로 다듬을 뿐 제거하지 않습니다(역노이즈를 적용하는 일 없이 유한한 개수의 랜덤화 샘플을 적용할 뿐입니다). 학습된 노이즈 딕셔너리도 층별 노이즈를 기술하기만 할 뿐, 2장에서는 그 역을 한 번도 쓰지 않았습니다. `inject_noise_strategy="no_modification"`에서 딕셔너리는 그냥 통과 전달이고, 이징 미러 회로의 $\langle Z_i\rangle$ 결과가 이상값 $+1$ 아래에 머무는 이유가 그것입니다. 회로가 깊어질수록 층별 레이트가 쌓여서, 완화하지 않은 편향 때문에 결과를 신뢰할 수 없는 지경에 이르고, 층 단위 오류 완화가 필요해집니다.

3장에서는 그다음 단계를 위한 기법들을 소개합니다. 이제 `inject_noise_strategy`를 바꿔야 합니다.
- PNA(`inject_noise_strategy=uniform_modification`)는 학습된 채널의 역을 *노이즈 완화 관측량*에 흡수합니다
- SLC(`inject_noise_strategy=individual_modification`)는 각 관측량의 *라이트콘*을 따라 반노이즈를 샘플링합니다

## 3장 — 오류 완화를 위한 Qiskit 애드온

3장에 오신 것을 환영합니다! 이 장에서는 2장에서 소개한 도구들로 고급 *층 단위* 오류 완화 기법을 구현합니다. 지금까지의 이야기를 정리해 봅시다. 1장에서는 `Estimator` 옵션으로 제공되는 완화 기법(`DD`, `PT`, `TREX`, `ZNE`, `PEA`, `PEC`)을 살펴봤습니다. 이들은 *회로 전체 스위치*입니다(`PEC`와 `PEA`는 *내부적으로는* 층 단위 완화를 하지만, API가 층 단위 손잡이를 드러내지 않습니다). 2장에서는 Samplomatic을 소개해서 회로를 박스로 묶고 오류 완화 결정을 특정 층으로 국한할 수 있게 됐습니다. 노이즈 학습도 2장에서 배웠습니다.

이 장에서는 그 재료들을 조합해, 회로 전체 기법을 넘어서는 구조 인지 애드온을 만듭니다. 세 가지 기법(그중 둘은 Qiskit 애드온)에 집중합니다:

- **3.1절: 전파 노이즈 흡수(PNA)** — 회로는 그대로 두고 관측량을 다시 써서 게이트 노이즈를 완화합니다.
- **3.2절: 확률적 오류 상쇄(PEC)** — 회로를 역노이즈로 다시 쓴 버전들을 샘플링해서 오류를 완화합니다.
- **3.2절: 음영 라이트콘(SLC)** — PEC의 아이디어를 선별적으로 적용해, 관측량에 영향을 줄 수 있는 것만 완화함으로써 샘플링 오버헤드를 줄입니다.

이 랩의 주인공은 *PNA*와 *SLC*입니다. 랩 제목이 가리키는 *양자 우위를 위한 새로운 도구*가 바로 이들입니다. 층 단위로 작동하면서 각 층 *자신의* 학습된 노이즈를 흡수하거나 상쇄하기 때문에, Runtime `Estimator` 옵션이 닿지 못하는 곳까지 가면서도 회로가 커져도 샘플링 오버헤드를 실용적인 수준으로 억제합니다.

오류 완화 기법 탐구에는 2장과 같은 1차원 횡자기장 이징 미러 회로를 사용합니다. 2.6.1절의 `construct_ising_circuit` 함수와 미러 트릭을 재사용하되, 층별 효과가 더 커져서 오류 완화 기법을 구현했을 때 그 효과가 잘 보이도록 사슬을 더 길게 키웁니다.

## 3.1 전파 노이즈 흡수(PNA)

이 절에서는 [**전파 노이즈 흡수(Propagated Noise Absorption, PNA)**](https://qiskit.github.io/qiskit-addon-pna/)를 소개합니다. 학습된 노이즈의 역을 회로가 아니라 관측량에 흡수시키는 오류 완화 기법입니다. 전통적인 확률적 오류 상쇄(PEC)는 노이즈 회로의 변형을 아주 많이 샘플링해야 하지만(3.2절에서 보게 됩니다), PNA는 그 오버헤드를 비켜 가면서 1장과 2장에서 쌓아 온 구조를 그대로 활용합니다. 드레싱된 층, 클리포드를 통한 파울리 전파, `Twirl`·`InjectNoise` 어노테이션이 붙은 박스 회로, 그리고 희소 파울리-린드블라드 생성원으로 학습된 노이즈까지 말이죠.

핵심 통찰은 이렇습니다. 노이즈가 파울리이거나 트월링으로 파울리 형태가 될 수 있다면, 그 역을 1.1.3절의 클리포드 켤레 규칙으로 회로를 따라 **앞으로 전파**시킬 수 있습니다. 그렇게 전파한 노이즈를 **측정 관측량에 흡수**시키면 *노이즈 완화 관측량*이 만들어집니다.

파울리 채널은 효율적으로 합성되고 전파되기 때문에, PNA는 노이즈 완화 관측량 $\tilde{O}$ 하나를 계산해 냅니다. 이 $\tilde{O}$를 노이즈 있는 회로에서 측정하면, 원래 원하던 관측량의 노이즈 없는 기댓값 $\langle O \rangle$의 추정치가 나옵니다.

이 절에서는 박싱, 학습, 전파, 흡수라는 PNA의 4단계 워크플로를 밟아 갑니다. 2장의 이징 미러 회로를 예제로 삼아 10큐비트, 트로터 2스텝으로 키운 다음, 이 예제로 PNA 워크플로 전체를 수행합니다.

### 3.1.1 설정과 10큐비트로 키우기

이 절에서도 2.6절에서 소개한 **1차원 횡자기장 이징 사슬** 해밀토니안을 계속 다룹니다:
$$
H = -J \sum_{\langle i,j \rangle} Z_i Z_j + h \sum_i X_i \;.
$$


10큐비트로 키우되, 트로터 2스텝과 회전각 π/8은 유지합니다. 아래에서는 문제 설정만 합니다. 이징 미러 회로를 만들고 트랜스파일합니다.

In [ ]:
# create 10 qubit settings
# ── Circuit ─────────────────────────────────────────────────────────
num_qubits_pna          = 10           # chain length
num_trotter_steps_pna   = 2            # Trotter "reps"
rx_angle           = np.pi / 8    # transverse-field rotation per step

# create the Ising chain circuit
ising_pna  = construct_ising_circuit(num_qubits_pna, num_trotter_steps_pna, rx_angle)
mirror_pna = ising_pna.compose(ising_pna.inverse())
mirror_pna.measure_all()

# transpile to obey ISA
mirror_isa_pna = isa_pm.run(mirror_pna)
layout_pna     = mirror_isa_pna.layout.final_index_layout()

print(f"Mirror     : {num_qubits_pna}q × {num_trotter_steps_pna} Trotter steps, "
      f"rx_angle = π/{round(np.pi / rx_angle)}")
print(f"ISA layout : physical qubits {layout_pna}")

### 3.1.2 회로 박싱과 노이즈 학습

새 이징 미러 회로를 박스로 묶기 전에, PNA용 박싱 패스 매니저를 새로 만들어야 합니다. 2장에서는 `inject_noise_strategy="no_modification"`인 박싱 패스 매니저를 썼습니다. 학습된 노이즈 모델을 samplex에 통과 전달로 넘길 뿐, 샘플링된 회로에 적용하지는 않는 설정이었습니다. 3장에서는 다른 전략이 필요합니다:

- **PNA**는 `inject_noise_strategy="uniform_modification"`을 씁니다. 샘플링 시점에 모든 층의 노이즈가 *같은 방식으로* 수정됩니다. PNA는 모든 `ref`의 `noise_scales`를 0으로 두는데, 이러면 샘플링된 회로는 건드리지 않으면서도 각 층이 자기 학습된 모델과 연결되어 있어서 노이즈 모델을 관측량으로 전파할 수 있습니다.
- **SLC**는 `inject_noise_strategy="individual_modification"`을 씁니다. 층마다(그리고 생성원마다) 독립적으로 수정할 수 있어서, SLC가 관측량의 라이트콘을 따라 생성원 단위로 노이즈를 스케일링할 수 있습니다. 3.2절에서 보게 됩니다.

`inject_noise_strategy` 선택에 대한 자세한 내용은 `generate_boxing_pass_manager` [문서](https://qiskit.github.io/samplomatic/api/auto/samplomatic.transpiler.generate_boxing_pass_manager.html#samplomatic.transpiler.generate_boxing_pass_manager/inject_noise_strategy)를 참고하세요.

새 박싱 패스 매니저를 정의한 뒤, `find_unique_box_instructions`로 회로의 고유 층을 찾습니다. 고유 층의 개수를 아래에 출력합니다.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PNA — boxing manager for uniform_modification + find unique layers
# ═══════════════════════════════════════════════════════════════════
pna_boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    measure_annotations="all",
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="uniform_modification",
)

# box the circuit for pna
boxed_circuit_pna = pna_boxing_pm.run(mirror_isa_pna)
# find the unique layers in the circuit
unique_layers_pna = find_unique_box_instructions(
    boxed_circuit_pna, normalize_annotations=None, undress_boxes=True,
)
print(f"PNA boxed  : {len(unique_layers_pna)} unique layers")

이제 회로의 노이즈를 학습할 준비가 됐습니다. 10큐비트 이징 회로는 앞서 노이즈를 학습했던 회로들과 다른 회로이므로, 여기서 다시 학습해야 합니다. 양자 컴퓨터의 노이즈는 시간이 지나면 변하기 때문에(캘리브레이션이 매일 이루어집니다), 이전 캘리브레이션 시기의 캐시된 학습 결과는 일반적으로 다시 학습하는 것이 맞습니다.

먼저 노이즈 학습기 옵션을 초기화합니다:

In [ ]:
# ── NoiseLearnerV3 new options (used by PNA, SLC, and indirectly by PEC) ───────
NL_NUM_RANDOMIZATIONS  = 16
NL_SHOTS_PER_RAND      = 32
NL_LAYER_PAIR_DEPTHS   = [1, 2, 4, 8]

pna_nl_options = NoiseLearnerV3Options(
    num_randomizations=NL_NUM_RANDOMIZATIONS,
    shots_per_randomization=NL_SHOTS_PER_RAND,
    layer_pair_depths=NL_LAYER_PAIR_DEPTHS,
)
pna_learner = NoiseLearnerV3(backend, pna_nl_options)

*참고:* 다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `NOISE_LEARN_JOB_ID_PNA` 파라미터에 붙여넣고 `SUBMIT_NOISE_JOB_PNA = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 15초입니다(ibm_pittsburgh 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
NOISE_LEARN_JOB_ID_PNA = None # paste job_id here on re-run
SUBMIT_NOISE_JOB_PNA   = False   # set True to submit a fresh learning job
pna_learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_PNA is not None:
    learner_job_pna = service.job(NOISE_LEARN_JOB_ID_PNA)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_PNA}")
elif SUBMIT_NOISE_JOB_PNA:
    learner_job_pna = pna_learner.run(unique_layers_pna)
    NOISE_LEARN_JOB_ID_PNA = learner_job_pna.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_PNA}")
else:
    print("Set SUBMIT_NOISE_JOB_PNA=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_PNA and re-run.")


In [ ]:
learner_job_pna = service.job(NOISE_LEARN_JOB_ID_PNA)
print(f"{NOISE_LEARN_JOB_ID_PNA}  (status: {learner_job_pna.status()})")

In [ ]:
if learner_job_pna.status() == "DONE":
    noise_learner_result_pna = learner_job_pna.result()
else:
    print(f"Not done yet (status={learner_job_pna.status()}). Re-run cell when DONE.")

고유 층마다 학습된 노이즈가 들어 있는 딕셔너리 `refs_to_noise_models_pna`를 꺼냅니다:

In [ ]:
if 'noise_learner_result_pna' in dir() and noise_learner_result_pna is not None:
    refs_to_noise_models_pna = noise_learner_result_pna.to_dict(unique_layers_pna, require_refs=False)
    print(f"refs_to_noise_models_pna has {len(refs_to_noise_models_pna)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


### 관심 관측량

횡자기장 이징 해밀토니안에는 ZZ 상호작용 에너지 항이 들어 있습니다. 관심 관측량으로 그런 항 하나, 가운데 두 큐비트에 걸린 항을 고릅니다:

$$
O \;=\;  Z_{4} Z_{5}, \;
$$

$N=10$ 큐비트 기준입니다. 최근접 이웃 ZZ 상관자입니다. 미러 회로의 이상적인 출력 $|0\rangle^{\otimes N}$에서는 모든 $\langle Z_i Z_{i+1}\rangle = +1$이므로, 이상적인 기댓값은 $\langle O\rangle = 1$입니다.

PNA는 관측량 $O$를 수정된 *노이즈 완화* 관측량 $\tilde{O}$로 다시 씁니다. 이 수정된 관측량에는 서로 다른 계수를 가진 추가 항들이 들어갑니다. $\tilde{O}$의 노이즈 있는 기댓값이 원래 $O$의 이상적인 기댓값과 일치하도록 만드는 것이 아이디어입니다.

먼저 관측량 $O$를 만들고, 회로의 레이아웃을 적용합니다.

In [ ]:
# create observable for the Ising model
# Single ZZ on the middle two qubits (1 term, ideal exp val = 1).
mid = num_qubits_pna // 2
observable_pna =  SparsePauliOp.from_sparse_list(
    [("ZZ", [mid - 1, mid], 1.0)], num_qubits=num_qubits_pna)

print(observable_pna)

In [ ]:
# apply layout to observable
observable_pna_isa = observable_pna.apply_layout(mirror_isa_pna.layout)

### 3.1.3  노이즈 완화 관측량 $\tilde{O}$ 계산하기

관측량 $O$를 만들었으니, PNA 워크플로의 다음 단계는 PNA 애드온의 함수 [`generate_noise_mitigating_observable`](https://qiskit.github.io/qiskit-addon-pna/apidocs/qiskit_addon_pna.html#qiskit_addon_pna.generate_noise_mitigating_observable)로 노이즈 완화 관측량 $\tilde{O}$를 계산하는 것입니다. 이 함수는 박스 회로, 관측량, 그리고 학습된 노이즈 딕셔너리를 받습니다.

각 `InjectNoise(ref=...)` 어노테이션은 노이즈 학습 단계에서 배운 자기 층의 `PauliLindbladMap`을 알고 있습니다. PNA는 그 채널들의 역을 회로를 따라 전파해서 관측량에 접어 넣습니다. 그 결과가 새 관측량 $\tilde{O}$이고, 그 노이즈 있는 기댓값은 원래 $O$의 이상적인 기댓값과 같습니다.

$\tilde{O}$가 측정 가능한 크기로 유지되도록 절단(truncation) 단계가 지배적인 항만 남깁니다. 관측량의 항이 많을수록 그 기댓값을 추정하는 데 필요한 회로 측정 횟수도 늘어나기 때문입니다. 절단은 `max_err_terms`와 `max_obs_terms`로 제어합니다.

박스 회로 자체는 바뀌지 않습니다. 바뀌는 것은 관측량뿐입니다.

In [ ]:
# PNA parameters
num_processes = 8
max_err_terms = 10000
max_obs_terms = 10000

# generate the noise mitigating observable
obs_tilde_isa = generate_noise_mitigating_observable(
    boxed_circuit_pna,
    observable_pna_isa,
    refs_to_noise_models_pna,
    max_err_terms=max_err_terms,
    max_obs_terms=max_obs_terms,
    num_processes=num_processes,
    print_progress=True,
    search_step=8,
)


In [ ]:
print(f"Number of Pauli terms in the noise mitigating observable: {len(obs_tilde_isa)}")

노이즈 완화 관측량 $\tilde{O}$를 논리 큐비트 공간에서 만들려면 후처리가 조금 필요합니다. 이렇게 해 두면 관측량을 들여다보며 어떤 항이 들어 있는지 확인하기 좋습니다.

In [ ]:
### Mapping the noise mitigating observable from physical to logical qubits

# Create an inverse layout: physical qubit index → logical qubit index
physical_to_logical = {
    physical_q: logical_q
    for logical_q, physical_q in enumerate(layout_pna)
}

# Extract the noise mitigating observable in sparse (Pauli, qubits, coeff) form
isa_pauli_terms = obs_tilde_isa.to_sparse_list()

# Remap each Pauli term from physical qubits to logical qubits
logical_pauli_terms = [
    (
        pauli_string,                                  # Pauli operators (e.g. "ZIX")
        [physical_to_logical[q] for q in phys_qubits], # physical → logical indices
        coefficient                                    # real-valued coefficient
    )
    for (pauli_string, phys_qubits, coefficient) in isa_pauli_terms
]

# Rebuild the observable in the logical (virtual) qubit space
obs_tilde_virtual = SparsePauliOp.from_sparse_list(
    logical_pauli_terms,
    num_qubits=num_qubits_pna
)

아래에서 $\tilde{O}$의 모든 항이 갖는 크기의 분포를 플롯합니다.

In [ ]:
# sort the observable magnitudes in descending order for plotting
obs_tilde_virtual = obs_tilde_virtual[np.argsort(np.abs(obs_tilde_virtual.coeffs))][::-1]

plt.figure(figsize=(8, 5))
plt.plot(np.abs(obs_tilde_virtual.coeffs), "o", color="tab:blue", markersize=6, alpha=0.7)
plt.yscale("log")
plt.xlabel("Pauli term index", fontsize=14)
plt.ylabel("Magnitude", fontsize=14)
plt.title(r"$\tilde{O}$ coefficient magnitudes", fontsize=14)
plt.grid(axis="both", alpha=0.3)
plt.tight_layout()
plt.show()

$\tilde{O}$에 항이 많고, 그중 상당수는 크기가 아주 작다는 것을 볼 수 있습니다. 크기순으로 정렬해서 처음 10개 항을 보고 구조를 살펴봅시다. 원래 관측량에는 항이 하나(가운데 두 큐비트의 ZZ)뿐이었다는 것을 기억하세요.

In [ ]:
### Truncating the noise mitigating observable
num_terms_to_plot = 10

# Take the absolute value of all coefficients
coeff_magnitudes = np.abs(obs_tilde_virtual.coeffs)

# Get indices that sort coefficients from smallest to largest
sorted_indices = np.argsort(coeff_magnitudes)

# Reverse the order to place largest coefficients first, reorder using sorted indices
sorted_indices_desc = sorted_indices[::-1]
obs_tilde_virtual = obs_tilde_virtual[sorted_indices_desc]

# Keep only the most significant Pauli terms
obs_tilde_virtual = obs_tilde_virtual[:num_terms_to_plot]


In [ ]:
def full_pauli_string(pstr, qubits, n):
    full = ["I"] * n
    for p, q in zip(pstr, qubits):
        full[q] = p
    return "".join(reversed(full))   # qubit 0 leftmost

# Sort by |coefficient| descending and keep top 10.
sorted_idx = np.argsort(np.abs(obs_tilde_virtual.coeffs))[::-1][:10]
obs_tilde_top = obs_tilde_virtual[sorted_idx]

# Identify which top-10 terms are originals vs PNA-added.
target_paulis_virtual = {
    full_pauli_string(p, q, obs_tilde_virtual.num_qubits)
    for p, q, _ in observable_pna.to_sparse_list()
}
labels = [
    full_pauli_string(p, q, obs_tilde_virtual.num_qubits)
    for p, q, _ in obs_tilde_top.to_sparse_list()
]
coeffs = np.abs(obs_tilde_top.coeffs)
is_orig = np.array([l in target_paulis_virtual for l in labels])

# Plot.
fig, ax = plt.subplots(figsize=(10, 4))
idx = np.arange(len(labels))
ax.plot(idx[is_orig],  coeffs[is_orig],  "s", color="C3", markersize=12, label="Original ZZ term")
ax.plot(idx[~is_orig], coeffs[~is_orig], "o", color="C0", markersize=10, label="PNA-added")
ax.set_yscale("log")
ax.set_xticks(idx)
ax.set_xticklabels(labels, rotation=45, ha="right", family="monospace")
ax.set_xlabel("Pauli term (virtual qubits)")
ax.set_ylabel(r"$|\tilde{c}|$")
ax.set_title(r"Top 10 terms in $\tilde{O}_{Z}$")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

원래의 ZZ 항이 가장 큰 크기를 갖는 것이 보일 겁니다. 추가된 항들은 회로의 노이즈를 감안하기 위한 것이므로 크기가 훨씬 작을 것으로 기대할 수 있습니다.

원래 관측량 $O$와 노이즈 완화 관측량 $\tilde{O}$의 항 크기를 직접 비교해 봅시다.

In [ ]:
# Extract Pauli strings and coefficients
pauli_strings = observable_pna.paulis.to_labels()
coeffs = observable_pna.coeffs

# Sort by coefficient magnitude
sorted_indices = sorted(
    range(len(coeffs)),
    key=lambda i: abs(coeffs[i]),
    reverse=True
)

# Print header
print("\n Original Observable Pauli Terms:\n")
print(f"{'Pauli string':<{observable_pna.num_qubits + 2}}  Coefficient")
print("-" * (observable_pna.num_qubits + 20))

# Print terms 
for i in sorted_indices[:num_terms_to_plot]:
    print(f"{pauli_strings[i]}  {coeffs[i].real:+.3f}")


In [ ]:
num_terms_to_plot = 10 

# Extract Pauli strings and coefficients
pauli_strings = obs_tilde_virtual.paulis.to_labels()
coeffs = obs_tilde_virtual.coeffs

# Sort by coefficient magnitude
sorted_indices = sorted(
    range(len(coeffs)),
    key=lambda i: abs(coeffs[i]),
    reverse=True
)

# Print header
print("\n Noise mitigating Observable Pauli Terms (top 10):\n")
print(f"{'Pauli string':<{obs_tilde_virtual.num_qubits + 2}}  Coefficient")
print("-" * (obs_tilde_virtual.num_qubits + 20))

# Print terms (no divider)
for i in sorted_indices[:num_terms_to_plot]:
    print(f"{pauli_strings[i]}  {coeffs[i].real:+.3f}")

가운데 ZZ 항은 원래 관측량에서 크기가 1이었습니다. $\tilde{O}$에서는 1보다 약간 커졌고 여전히 지배적인 항입니다. PNA가 원래 항을 증폭하고, 층별 노이즈를 흡수할 새 항들을 추가한 것입니다. 위의 표는 가장 큰 10개 항을 보여 줍니다.

결과는 사용하는 QPU와 그로부터 학습된 노이즈에 따라 달라집니다. 노이즈는 드리프트하고 캘리브레이션은 매일 이루어집니다. 같은 QPU라도 노이즈는 시간이 지나면 변합니다. 가장 정확한 결과를 얻으려면 최종 작업을 돌리기 직전에 항상 노이즈를 다시 학습하는 것이 좋습니다.

### Exercise 4 — 자화 관측량 완화하기

<div class="alert alert-block alert-success">

<b>Exercise 4 — 자화 관측량 완화하기</b>

이 연습 문제에서는 10큐비트 이징 사슬의 **자화**(magnetization) 관측량

$$
O_Z = \sum_{i=0}^{N-1} Z_i, \;
$$

에 대해 PNA 노이즈 완화 관측량을 만들어야 합니다. 큐비트마다 $Z$ 하나씩을 더한 합입니다. 미러의 이상적인 출력 $|0\rangle^{\otimes N}$에서는 모든 $\langle Z_i \rangle = +1$이므로, 10큐비트 사슬의 이상적인 기댓값은 $\langle O_Z \rangle = 10$입니다.

PNA는 각 층의 학습된 노이즈를 회로가 아니라 관측량에 흡수시킵니다. 여기서는 10큐비트 박스 미러 위에서 노이즈 완화 자화 관측량 $\tilde{O}_Z$를 만든 다음, PNA가 그 안으로 전파해 넣은 반노이즈 항들을 살펴봅니다.

노이즈 완화 관측량은 세 단계로 만듭니다:

- **목표:** 10큐비트에서 $O_Z = \sum_{i=0}^{9} Z_i$에 해당하는 `SparsePauliOp`를 `target_observable_ex4`로 정의하세요. `SparsePauliOp.from_sparse_list`에 큐비트당 항목 하나씩을 넣으면 됩니다.
- **레이아웃:** PNA는 물리 큐비트 인덱스로 노이즈 층을 맞추므로, `apply_layout(mirror_isa_pna.layout)`으로 관측량을 ISA에 매핑하세요 → `target_observable_ex4_isa`
- **완화:** ISA 관측량에 `generate_noise_mitigating_observable`을 호출하되, 스텁에 미리 정해 둔 절단 파라미터들을 키워드 인자로 넘기세요 → `obs_tilde_ex4`

채점기는 목표 관측량과 노이즈 완화 관측량을 확인합니다.

</div>

In [ ]:
#TODO: Add Your code here and set the missing values.


target_observable_ex4     = None
target_observable_ex4_isa = None
obs_tilde_ex4             = None

# PNA parameters (same as in section 3.1.3 main text).
num_processes = 8
max_err_terms = 10000
max_obs_terms = 10000

# 1. Build target_observable_ex4 as a SparsePauliOp with 10 Z terms one
#    on each qubit
target_observable_ex4 = ...

# 2. Apply mirrored_isa_pna.layout to the target so PNA receives an ISA-mapped observable.
target_observable_ex4_isa = ...

# 3. Call generate_noise_mitigating_observable(boxed_circuit_pna, target_observable_ex4_isa,
#    refs_to_noise_models_pna, max_err_terms=..., max_obs_terms=..., num_processes=...).
obs_tilde_ex4 = ...

In [ ]:
# grade your answer
grade_lab3_ex4(
    target_observable_ex4,
    target_observable_ex4_isa,
    obs_tilde_ex4
)

####  $\tilde{O}_{Z}$는 어떻게 생겼을까?

아래 플롯은 $\tilde{O}_{Z}$에서 가장 큰 20개 항을 가상 큐비트 기준 파울리 문자열과 함께 보여 줍니다. 원래의 10개 항(빨간 사각형)이 약간 증폭된 채 맨 위에 자리 잡은 것이 보일 겁니다. 파란 원들은 PNA가 학습된 노이즈 모델로부터 전파해 넣은 반노이즈 보정입니다.

In [ ]:
# Map obs_tilde back to virtual qubits for readable Pauli labels.
p_2_v = {p: v for v, p in enumerate(layout_pna)}
obs_tilde_virtual_ex4 = SparsePauliOp.from_sparse_list(
    [(p, [p_2_v[q] for q in qs], c) for p, qs, c in obs_tilde_ex4.to_sparse_list()],
    num_qubits=num_qubits_pna,
)

# Sort by |coefficient| descending and keep top 20.
sorted_idx = np.argsort(np.abs(obs_tilde_virtual_ex4.coeffs))[::-1][:20]
obs_tilde_top = obs_tilde_virtual_ex4[sorted_idx]

def full_pauli_string(pstr, qubits, n):
    full = ["I"] * n
    for p, q in zip(pstr, qubits):
        full[q] = p
    return "".join(reversed(full))   # qubit 0 leftmost


# Identify which top-20 terms are originals vs PNA-added.
target_paulis_virtual = {
    full_pauli_string(p, q, num_qubits_pna)
    for p, q, _ in target_observable_ex4.to_sparse_list()
}
labels = [full_pauli_string(p, q, num_qubits_pna)
          for p, q, _ in obs_tilde_top.to_sparse_list()]
coeffs = np.abs(obs_tilde_top.coeffs)
is_orig = np.array([l in target_paulis_virtual for l in labels])

# Plot.
fig, ax = plt.subplots(figsize=(10, 4))
idx = np.arange(len(labels))
ax.plot(idx[is_orig],  coeffs[is_orig],  "s", color="C3", markersize=12, label="Original observable terms")
ax.plot(idx[~is_orig], coeffs[~is_orig], "o", color="C0", markersize=10, label="PNA-added")
ax.set_yscale("log")
ax.set_xticks(idx)
ax.set_xticklabels(labels, rotation=45, ha="right", family="monospace")
ax.set_xlabel("Pauli term (virtual qubits)")
ax.set_ylabel(r"$|\tilde{c}|$")
ax.set_title(r"Top 20 terms in $\tilde{O}$")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### 3.1.4  하드웨어에서 실행하기

$\tilde{O}$가 준비됐으니 Executor 프로그램을 만들어 제출합니다.

이 절에는 다음이 필요합니다:
- `boxed_circuit_pna` (uniform_modification으로 박싱한 것)
- Exercise 4에서 만든 `obs_tilde_ex4` (자화 관측량용)
- 10큐비트용 노이즈 학습기에서 얻은 `refs_to_noise_models_pna`

3.1.5절에서는 세 경우의 기댓값 결과를 비교합니다. 완화하지 않은 관측량, PNA 워크플로의 노이즈 완화 관측량, 그리고 PNA + TREX의 노이즈 완화 관측량입니다.

In [ ]:
# Build template circuit and samplex for use with the "Executor"
template_circuit_pna, samplex_pna = samplomatic.build(boxed_circuit_pna)

In [ ]:
# number of terms in noise mitigating observable to truncate to
num_to_measure = 10

# Identify measured qubits in canonical (measurement) order.
meas_box = boxed_circuit_pna.data[-1]
canonical_qubits = [
    i for i, q in enumerate(boxed_circuit_pna.qubits)
    if q in meas_box.qubits
]

# Build qubit index mappings: canonical → physical → virtual (logical).
canonical_to_physical = dict(enumerate(canonical_qubits))
physical_to_virtual   = {p: v for v, p in enumerate(layout_pna)}
canonical_to_virtual  = {
    c: physical_to_virtual[p]
    for c, p in canonical_to_physical.items()
}

# Discover the ChangeBasis ref from the measurement box rather than hard-coding
# the value below (samplomatic auto-assigns refs at boxing time and the value
# depends on the circuit — e.g. "basis2" in some configurations, "basis0" in
# others). The dynamic lookup keeps the rest of this section circuit-agnostic.
change_basis_ann = get_annotation(meas_box.operation, ChangeBasis)
if change_basis_ann is None:
    raise RuntimeError(
        "No ChangeBasis annotation on the measurement box. "
        "The boxing pass manager must be built with measure_annotations='all'."
    )
basis_ref = change_basis_ann.ref
print(f"ChangeBasis ref: {basis_ref!r}")

# Truncate Õ_Z to the largest terms before measurement (lower variance per shot).
obs_tilde_virtual_ex4 = obs_tilde_virtual_ex4[
    np.argsort(np.abs(obs_tilde_virtual_ex4.coeffs))[::-1]
][:num_to_measure]

# Bases needed to cover every Pauli term in the truncated Õ_Z.
meas_bases, bases_reverser = get_measurement_bases(obs_tilde_virtual_ex4)
meas_bases_canonical = [
    np.array(
        [base[canonical_to_virtual[c]] for c in sorted(canonical_to_virtual)],
        dtype=np.uint8,
    )
    for base in meas_bases
]

# Encoding inside the canonical arrays: I=0, Z=1, X=2, Y=3.
# Always include the all-Z basis so we can also extract an *unmitigated* <O_Z>
# from the same execution. The truncated Õ_Z may not naturally need all-Z
# (e.g. if PNA-added X/Y terms dominate the truncation), but adding it costs
# only one extra basis worth of shots and keeps 3.1.5 self-contained.
all_z_canonical = np.ones(num_qubits_pna, dtype=np.uint8)

if any(np.array_equal(b, all_z_canonical) for b in meas_bases_canonical):
    all_z_idx = next(
        i for i, b in enumerate(meas_bases_canonical)
        if np.array_equal(b, all_z_canonical)
    )
    print(f"all-Z basis already present at index {all_z_idx}")
else:
    meas_bases_canonical.append(all_z_canonical)
    all_z_idx = len(meas_bases_canonical) - 1
    print(f"Appended all-Z basis at index {all_z_idx}")

# Decoded bases printout (qubit 0 leftmost) — useful for sanity checks.
def _decode(b):
    return "".join("IZXY"[v] for v in b)

print(f"\nMeasurement bases (canonical order, qubit 0 leftmost):")
for i, b in enumerate(meas_bases_canonical):
    note = "  ← unmitigated baseline" if i == all_z_idx else ""
    print(f"  [{i}] {_decode(b)}{note}")


#### 추가 확인: 항 개수 vs. 측정 기저 개수

위의 절단은 항의 *개수*를 고정해서 남기지만, 샷 비용을 좌우하는 양은 서로 다른 **측정 기저**의 개수입니다. `get_measurement_bases`는 큐비트별로 교환하는 파울리 항들을 묶어서 기저를 공유하게, 따라서 샷도 공유하게 만듭니다. PNA가 추가한 항이라도 이미 측정하는 기저에서 대각이라면, 추가 *기저* 비용 없이 남겨 둘 수 있습니다.

아래 셀은 전체 $\tilde{O}_Z$를 다시 만들고, `num_to_measure` 컷 바로 바깥의 항들 중 몇 개가 이미 필요한 기저 집합에 들어오는지 셉니다. 그런 항을 남기는 것은 측정 기저 관점에서 "공짜"입니다. 물론 통계적으로 완전히 공짜는 아닙니다. 추가되는 항 하나하나가 자기 계수와 샷 노이즈를 $\langle\tilde{O}_Z\rangle$의 합산 추정치에 보태니까요. 절약되는 것은 새 기저, 즉 새 샷 배분이 필요 없다는 점입니다.

In [ ]:
# ── Aside: term count vs. measurement-basis count ───────────────────────────────
#
# Rebuild the *full* O_Z-tilde (obs_tilde_virtual_ex4 was truncated in place above)
# so we can sweep past the cut and see which extra terms are "free".
p_2_v = {p: v for v, p in enumerate(layout_pna)}
obs_tilde_virtual_full = SparsePauliOp.from_sparse_list(
    [(p, [p_2_v[q] for q in qs], c) for p, qs, c in obs_tilde_ex4.to_sparse_list()],
    num_qubits=num_qubits_pna,
)
obs_full_sorted = obs_tilde_virtual_full[
    np.argsort(np.abs(obs_tilde_virtual_full.coeffs))[::-1]
]

# Bases required by the current truncation.
bases_cut, _ = get_measurement_bases(obs_full_sorted[:num_to_measure])
n_bases_cut  = len(bases_cut)
print(f"Truncation : {num_to_measure} terms  ->  {n_bases_cut} measurement bases")

# Walk the terms just past the cut: a term is "free" if it needs no new basis.
free = 0
for k in range(num_to_measure + 1, len(obs_full_sorted) + 1):
    bases_k, _ = get_measurement_bases(obs_full_sorted[:k])
    if len(bases_k) == n_bases_cut:
        free += 1
    else:
        break  # first term that would require a new basis

if free:
    print(f"Past cut   : {free} further term(s) fit the existing {n_bases_cut} "
          f"bases -- keeping {num_to_measure + free} terms costs no extra basis.")
else:
    print("Past cut   : the next term needs a new basis -- no free terms here.")
print("(Free in *basis* cost; each extra term still adds its own shot noise.)")


In [ ]:
# Control the # of shots during execution
shots_per_randomization_exec = 64
num_randomizations_exec = 1024

# Zero out the noise so it isn't actually injected during execution.
# We only added InjectNoise annotations so PNA could associate the noise
# to layers in the circuit.
samplex_inputs_pna = {f"noise_scales.{ref}": 0.0 for ref in refs_to_noise_models_pna}
samplex_inputs_pna |= {"pauli_lindblad_maps": refs_to_noise_models_pna}

# Specify the bases to measure.
# The ref name "basis2" is what `samplomatic.build` auto-assigned to the
# measurement-box ChangeBasis annotation for this particular circuit.
# To confirm the ref name for any other circuit, run:
#     print(samplex_pna.inputs().make_broadcastable().describe())
# and look for the line beginning with "basis_changes.<ref>".

bases_broadcastable = np.expand_dims(np.array(meas_bases_canonical), axis=1)

#added to accommodate auto generated basis_changes
change_basis_ann = get_annotation(meas_box.operation, ChangeBasis)
basis_ref = change_basis_ann.ref

samplex_inputs_pna |= {"basis_changes": {basis_ref: bases_broadcastable}}

In [ ]:
# Convert samplex_inputs into a dict to pass to QuantumProgram.
samplex_arguments_pna = samplex_pna.inputs().make_broadcastable().bind(**samplex_inputs_pna)

# Instantiate the QuantumProgram with the specified parameters.
program = QuantumProgram(shots=shots_per_randomization_exec)
program.append_samplex_item(
    circuit=template_circuit_pna,
    samplex=samplex_pna,
    samplex_arguments=samplex_arguments_pna,
    shape=(num_randomizations_exec,),
)

#### Executor 작업 실행하기

*참고:* 다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `EXECUTOR_JOB_ID_PNA` 파라미터에 붙여넣고 `SUBMIT_EXECUTOR_JOB_PNA = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 30초입니다(ibm_fez 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
executor = Executor(backend)

EXECUTOR_JOB_ID_PNA      = None      # paste an Executor job_id here on re-run
SUBMIT_EXECUTOR_JOB_PNA  = False    # set True to submit a fresh Executor job
executor.options.environment.job_tags = ["qgss26"]

if EXECUTOR_JOB_ID_PNA is not None:
    job_exec_pna = service.job(EXECUTOR_JOB_ID_PNA)
    print(f"Re-using saved job: {EXECUTOR_JOB_ID_PNA}")
elif SUBMIT_EXECUTOR_JOB_PNA:
    job_exec_pna = executor.run(program)
    EXECUTOR_JOB_ID_PNA = job_exec_pna.job_id()      
    print(f"Submitted: {EXECUTOR_JOB_ID_PNA}")
else:
    print("Set SUBMIT_EXECUTOR_JOB_PNA=True to submit a fresh job, "
          "or paste a saved Executor job id into EXECUTOR_JOB_ID_PNA and re-run.")

In [ ]:
# Verify the job type and status
if EXECUTOR_JOB_ID_PNA is not None:
    print(f"Program id  : {job_exec_pna.job_id()}")
    print(f"Status      : {job_exec_pna.status()}")

작업 결과를 꺼냅니다:

In [ ]:
if job_exec_pna.status() == "DONE":
    exec_results_pna = job_exec_pna.result()
else:
    print(f"Not done yet (status={job_exec_pna.status()}). Re-run cell when DONE.")

#### TREX 오류 완화 기법 얹기

1.1.1절과 1.1.2절에서 [트월링 기반 판독 오류 제거(TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620)를 `Estimator` 프리미티브에서 쉽게 켤 수 있는 오류 완화 기법으로 소개했습니다. TREX는 측정 직전에 무작위로 $X$ 게이트를 삽입하고 결과를 고전적으로 뒤집는 방식으로 측정의 판독 오류를 완화합니다. 이렇게 하면 판독 오류 행렬이 대각화되어 판독 오류를 완화할 수 있게 됩니다.

여기 PNA 워크플로에서는 [Qiskit 애드온 유틸리티 라이브러리](https://github.com/Qiskit/qiskit-addon-utils)의 함수 `trex_factors`로 TREX를 손쉽게 구현할 수 있습니다.

In [ ]:
# Computing the TREX factors
measurement_noise_map = noise_learner_result_pna[2].to_pauli_lindblad_map()
trex_rescale_factors = trex_factors(measurement_noise_map, bases_reverser)

### 3.1.5  결과 분석하기

이 절에서는 자화 기댓값의 세 가지 추정치를 비교합니다. 완화하지 않은 것, PNA만 쓴 것, PNA + TREX입니다. 미러의 출력 상태 $|0\rangle^{\otimes N}$에서 기댓값의 이상적인 값은 $N=10$입니다.

세 경우 모두 Qiskit 애드온 유틸리티 라이브러리의 함수 `executor_expectation_values`로 기댓값을 계산합니다.

PNA만으로 오류 완화한 기댓값부터 시작해 봅시다.

In [ ]:
# Expectation value with PNA only.
# `bases_reverser` was constructed from `obs_tilde_virtual_ex4`, so it pairs
# each measurement basis to the (post-PNA) Pauli terms diagonal in that basis.
#
# The data axis-0 length is len(meas_bases_canonical), which may be one larger
# than len(bases_reverser) — we appended an all-Z basis for the unmitigated
# baseline. Slice the data to the original PNA bases before passing in.

n_pna_bases = len(bases_reverser)

results_pna = executor_expectation_values(
    exec_results_pna[0]["meas"][:n_pna_bases],
    bases_reverser,
    meas_basis_axis=0,
    avg_axis=1,
    measurement_flips=exec_results_pna[0]["measurement_flips.meas"][:n_pna_bases],
    pauli_signs=None,  # noise_scales=0 ⇒ no PEC ⇒ no Pauli signs
    rescale_factors=None,
)

exp_val_pna = results_pna[0][0]
std_pna     = results_pna[0][1]
print(f"PNA only    : <O_Z> = {exp_val_pna:+.4f}  std = {std_pna:.4e}")


다음으로 완화하지 않은 노이즈 섞인 기댓값을 계산합니다:

In [ ]:
# ─── Unmitigated expectation value ───
# We added the all-Z basis to meas_bases_canonical above (index `all_z_idx`),
# so the executor measured it as part of the same job. Slice that subset of
# the data and pair it with the original observable O_Z.

meas_z  = exec_results_pna[0]["meas"][all_z_idx]
flips_z = exec_results_pna[0]["measurement_flips.meas"][all_z_idx]

bases_reverser_unmit = {Pauli("Z" * num_qubits_pna): [target_observable_ex4]}

res_unmit = executor_expectation_values(
    meas_z,
    bases_reverser_unmit,
    meas_basis_axis=None,   # already sliced — no basis axis remaining
    avg_axis=0,              # average over randomizations (now axis 0)
    measurement_flips=flips_z,
    pauli_signs=None,        # noise_scales=0 ⇒ no PEC ⇒ no Pauli signs
    rescale_factors=None,
)
exp_val_unmit, std_unmit = res_unmit[0][0], res_unmit[0][1]
print(f"Unmitigated : <O_Z> = {exp_val_unmit:+.4f}  std = {std_unmit:.4e}")


마지막으로 PNA와 TREX 두 오류 완화 기법을 모두 적용한 기댓값을 계산합니다:

In [ ]:
# ─── PNA + TREX ───
# TREX rescales the data per-basis to absorb the readout (measurement) noise
# learned by NoiseLearnerV3's measurement-layer model. As above, slice the
# data to the original PNA bases (skipping the appended all-Z basis used only
# for the unmitigated baseline).
measurement_noise_map = noise_learner_result_pna[2].to_pauli_lindblad_map()
trex_rescale_factors  = trex_factors(measurement_noise_map, bases_reverser)

n_pna_bases = len(bases_reverser)

results_pna_trex = executor_expectation_values(
    exec_results_pna[0]["meas"][:n_pna_bases],
    bases_reverser,
    meas_basis_axis=0,
    avg_axis=1,
    measurement_flips=exec_results_pna[0]["measurement_flips.meas"][:n_pna_bases],
    pauli_signs=None,  # noise_scales=0 ⇒ no PEC ⇒ no Pauli signs
    rescale_factors=trex_rescale_factors,
)

exp_val_pna_trex = results_pna_trex[0][0]
std_pna_trex     = results_pna_trex[0][1]
print(f"PNA + TREX  : <O_Z> = {exp_val_pna_trex:+.4f}  std = {std_pna_trex:.4e}")

결과를 플롯합니다:

In [ ]:
# Compare: Unmitigated, PNA, PNA+TREX vs. ideal (= N = 10 for the 10q chain).
experiments = ["Unmitigated", "PNA", "PNA+TREX"]
colors      = ["tab:gray", "tab:blue", "tab:orange"]
markers     = ["o", "s", "^"]

evs  = [exp_val_unmit, exp_val_pna, exp_val_pna_trex]
errs = [std_unmit,     std_pna,     std_pna_trex]
x    = np.arange(len(experiments))

plt.figure(figsize=(6, 4))
for xi, yi, ei, label, color, marker in zip(x, evs, errs, experiments, colors, markers):
    plt.errorbar(
        xi, yi, yerr=ei,
        color=color, marker=marker, markersize=12,
        linestyle="none", capsize=5,
        label=label, zorder=3,
    )

plt.axhline(y=num_qubits_pna, color="green", linestyle="--", linewidth=2,
            label=f"Ideal = N = {num_qubits_pna}", zorder=2)

plt.xticks(x, experiments)
plt.ylabel("Expectation value", fontsize=14)
plt.title(r"10 qubit Ising chain, 2 Trotter steps, $O_Z$ obs",
          fontsize=14)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


위의 플롯에서 세 구현 사이의 뚜렷한 차이가 보일 겁니다. 완화하지 않은 기댓값이 가장 나빠야 합니다(즉 이상값 10에서 가장 멀어야 합니다). 완화하지 않은 관측량 $O$ 대신 노이즈 완화 관측량 $\tilde{O}$의 기댓값을 측정하는 방식으로 PNA를 구현하면 결과가 좋아져야 하고, 여기에 TREX까지 더해 판독 오류도 함께 완화하면 결과가 한 번 더 좋아지는 것을 볼 수 있어야 합니다.

이것으로 전파 노이즈 흡수(PNA)를 다룬 3.1절을 마칩니다. 3.2절에서는 PEC와 SLC로 넘어갑니다.

## 3.2 확률적 오류 상쇄(PEC)와 음영 라이트콘(SLC)

### 3.2.1 확률적 오류 상쇄(PEC) 개요
[확률적 오류 상쇄(PEC)](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)는 회로 수준에서 작동합니다. 2장에서 학습한 파울리 노이즈 모델로 역노이즈 맵(반노이즈)을 구성한 다음, 준확률(quasi-probability) 가중치에서 나오는 샷별 부호와 함께 그 역에서 샘플링합니다. 마지막에 $\gamma$로 스케일을 다시 맞추면, 노이즈 모델이 정확하다는 전제 아래 평균낸 기댓값은 이상적인 회로에 대해 비편향입니다.

개념적으로 PEC는 PNA의 회로 쪽 대응물입니다:
- **PNA**는 회로를 고정하고 관측량을 다시 씁니다.
- **PEC**는 관측량을 고정하고 회로를 확률적으로 다시 씁니다.

PEC는 범용적입니다. 노이즈 모델만 정확하다면 관측량에 특별한 구조를 요구하지 않고 오류를 완화합니다. 이 범용성의 대가는 추정량 분산의 증가입니다. 필요한 샷 수는 회로 깊이와 총 노이즈에 따라 늘어나며, 오버헤드 인자 $\gamma^2$로 요약됩니다. 정량적으로는 $$\gamma = \exp\!\big(2\sum_{l,\sigma}\lambda_{l,\sigma}\big),$$ 이고, $\lambda_{l,\sigma}$는 회로의 층 $l$에서 생성원 $\sigma$의 학습된 레이트입니다. 샘플링 비용은 $\gamma^2$로 자랍니다.

1.1.2절에서 봤듯 Runtime `Estimator`는 플래그 하나짜리 PEC 옵션(`resilience.pec_mitigation = True`)을 제공합니다. 이 스위치는 편리하고 *내부적으로는* 층별 노이즈 모델을 쓰지만, 층 단위·생성원 단위 손잡이를 사용자에게 드러내지 않으며, 학습된 생성원을 전부 상쇄하기 때문에 샘플링 오버헤드 $\gamma^2$가 깊이에 따라 지수적으로 폭발할 수 있습니다.

이 절에서는 **음영 라이트콘**(SLC) 애드온으로 $\gamma^2$를 유틸리티 규모 회로에서도 감당할 만한 수준으로 끌어내리는 데 집중합니다. SLC가 어떻게 전체 PEC 대비 샘플링 오버헤드를 줄이는지 이어지는 절들에서 보게 됩니다. 세 가지 다른 관측량에 대해 전체 PEC와 SLC 애드온의 샘플링 오버헤드를 비교해 볼 것입니다.

### 3.2.2 음영 라이트콘(SLC) 개요

[음영 라이트콘(SLC)](https://qiskit.github.io/qiskit-addon-slc/)은 PEC의 진화형입니다. 회로의 *모든* 노이즈를 상쇄하는 대신, 회로의 각 노이즈 생성원이 관심 관측량에 얼마나 영향을 줄 수 있는지 그 한계를 계산합니다. 영향의 한계가 작은 생성원은 스케일을 줄이거나 건너뛰어서, 같은 잔여 편향에서 PEC의 샘플링 오버헤드 $\gamma^2$를 줄입니다.

직관은 인과성에서 옵니다. 회로 끝에서 측정하는 관측량은 자기 역방향 라이트콘 *안의* 노이즈, 즉 교란이 측정까지 전파될 수 있는 위치들의 노이즈에만 영향을 받을 수 있습니다. 라이트콘 밖의 오류는 측정 결과에 영향을 줄 수 없으므로 오류 상쇄 절차에서 제외해도 됩니다. SLC는 안이냐 밖이냐로만 나누는 이분법적 라이트콘에서 한 발 더 나아갑니다. 각 노이즈 생성원에 (이분법적 안/밖 대신) 관측량을 얼마나 세게 흔들 수 있는지에 대한 한계값을 부여합니다. 바탕이 되는 방법은 [Lightcone Shading for Classically Accelerated Quantum Error Mitigation](https://arxiv.org/abs/2409.04401)을 참고하세요.

SLC의 워크플로는 다음과 같습니다:

1. **순방향 바운드** 계산: 각 노이즈 생성원이 존재한다면 앞으로 전파되어 최종 관측량에 어떻게 영향을 줄지를 정의합니다.

2. **역방향 바운드** 계산: 관측량을 뒤로 전개했을 때 각 노이즈 위치에서 기여를 어떻게 얻는지를 정의합니다.

3. **병합**: 순방향·역방향 바운드를 학습된 노이즈 모델과 병합해서 오류 생성원별 관련도 점수를 얻습니다.

4. 그 점수를 `compute_local_scales` 함수에 넘겨 어떤 오류를 완화할 가치가 있는지 고릅니다. 통제된 편향을 받아들이는 대신 더 작은 $\gamma^2$를 얻습니다.

#### SLC 애드온 임포트:

In [ ]:
# All SLC add-on imports were already loaded at the top of the notebook in Setup & Imports

# Quick sanity check that we have all the necessary imports:
required = [
    "compute_backward_bounds", "compute_forward_bounds", "compute_local_scales",
    "merge_bounds", "generate_noise_model_paulis", "map_modifier_ref_to_ref",
    "draw_shaded_lightcone",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise ImportError(
        f"Missing SLC names: {missing}. Re-run the master imports cell at the "
        f"top of the notebook."
    )
print("SLC imports OK.")


### 3.2.3 설정과 국소 관측량

SLC의 효과를 보려면 라이트콘이 관찰될 만큼 깊은 예제 회로가 필요합니다. 그래서 이징 미러 회로 예제를 이어 가되 트로터 스텝 수를 10으로 늘립니다. 스텝이 많을수록 회로가 깊어지고, 라이트콘도 깊어집니다.

- `num_qubits_pna` $= 10$
- 트로터 스텝 수를 10으로 늘립니다(`num_trotter_steps_slc` $= 10$)
- `rx_angle` = $\pi / 8$은 유지합니다

*참고:* SLC는 3.1절에서 PNA에 썼던 것과 다른 노이즈 주입 전략을 요구합니다. PNA의 `uniform_modification`은 전역 `noise_scales` 슬롯 하나만 드러내지만, SLC는 관측량의 라이트콘을 따라 파울리 생성원 하나하나를 스케일링해야 하므로 *생성원 단위* 제어가 필요합니다. SLC의 `inject_noise_strategy="individual_modification"`은 `InjectNoise` 어노테이션마다 별도의 `noise_scales.<ref>` 슬롯을 드러냅니다.

In [ ]:
# new boxing pass manager for SLC with individual modification
slc_boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    measure_annotations="all",
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="individual_modification",
)

### 관심 관측량

이 절에서 다룰 관측량은 국소 관측량 $O = Z_4$, 즉 10큐비트 이징 사슬에서 큐비트 4에 걸린 $Z$입니다. 10개 큐비트 중 1개에만 작용하므로 이 관측량은 *국소적*(local)입니다.

In [ ]:
# Local observable: Z on central qubit and identity elsewhere.
target_obs_local: list[tuple[str, list[int], float]] = [("Z", [4], 1.0)]

# Build the observable.
observable_local = SparsePauliOp.from_sparse_list(target_obs_local, num_qubits=num_qubits_pna)
print(observable_local)
 
# Build the measurement bases. 
bases_virt, reverser_virt = get_measurement_bases(observable_local)

#### SLC는 왜 국소 관측량에 잘 맞을까?

SLC 방법이 국소 관측량에 특히 잘 맞는 이유는 양자 회로의 인과 구조를 활용하기 때문입니다. 국소 관측량은 소수의 큐비트에만 작용하므로, SLC는 관련된 라이트콘 — 관측량의 기댓값에 인과적으로 영향을 주는 게이트들의 부분집합 — 을 효율적으로 찾아내 거기에 집중할 수 있습니다. 라이트콘 밖의 오류는 안심하고 무시할 수 있어서 계산 오버헤드가 크게 줄어듭니다. 이런 표적화 덕분에, 모든 오류를 추적해야 하는 방법들에 비해 SLC는 국소 관측량에서 더 효율적입니다.

반대로 10개 큐비트 전부에 작용하는 관측량을 생각하면, 해당 라이트콘은 회로의 더 넓은 부분을 뒤덮게 됩니다. 그만큼 SLC가 더 많은 게이트의 오류를 추적해야 하니, PEC 단독 대비 샘플링 오버헤드 절감이 크지 않을 수 있습니다.

Exercise 5에서는 국소성 정도가 다른 세 가지 관측량을 다루면서, 관측량의 국소성이 SLC의 샘플링 오버헤드에 미치는 영향을 직접 확인하게 됩니다.

국소 관측량을 정의했으니, 이 랩에서 이미 여러 번 밟았던 단계를 따라 문제를 설정합니다:

1. 이징 회로 만들기 (이번에는 트로터 10스텝)
2. 회로를 미러로 만들고 끝에서 모든 큐비트 측정하기
3. 미러 회로를 백엔드 ISA에 맞게 트랜스파일하기
4. 관측량을 ISA 레이아웃에 매핑하기
5. 회로를 박스로 묶고 고유 층 찾기

이 다섯 단계를 아래 셀에서 전부 구현합니다.

In [ ]:
num_trotter_steps_slc = 10 # increase to 10 to see the light cone

# Build the mirrored Ising circuit.
ising_slc = construct_ising_circuit(num_qubits_pna, num_trotter_steps_slc, rx_angle)
mirror_slc = ising_slc.compose(ising_slc.inverse())
mirror_slc.measure_all()

# Transpile to the backend ISA and capture the layout.
mirror_isa_slc = isa_pm.run(mirror_slc)

# Map observable to ISA layout 
observable_local_isa = observable_local.apply_layout(mirror_isa_slc.layout, num_qubits=mirror_isa_slc.num_qubits)
layout_slc = mirror_isa_slc.layout.final_index_layout()
wire_order_slc = layout_slc + [q for q in range(mirror_isa_slc.num_qubits) if q not in layout_slc]

# Box for SLC (uses `individual_modification` so we can scale per-error-term later).
boxed_circuit_slc = slc_boxing_pm.run(mirror_isa_slc)
unique_layer_slc = find_unique_box_instructions(
    boxed_circuit_slc, normalize_annotations=None, undress_boxes=True
)

### 3.2.4 학습될 노이즈 모델 파울리 예측하기

[Qiskit SLC 애드온](https://qiskit.github.io/qiskit-addon-slc/)의 함수 `generate_noise_model_paulis`로 회로의 파울리 수준 노이즈 기술을 구성합니다:

- 이 단계는 노이즈 학습에 대한 어떤 지식도 *없이* 수행됩니다.
- 함수는 회로의 박스 층을 하나씩 훑습니다.
- 관련된 **가중치 1**과 **가중치 2**의 파울리 노이즈 항을 모두 생성합니다.
- 파울리 항은 회로의 큐비트 연결성으로 제한됩니다(활성 큐비트와 간선 위에 지지되는 항만 남깁니다)
- 이렇게 얻은 파울리 집합은 노이즈 세기와 무관하게, 노이즈가 있을 수 있는 자리를 정의합니다.
- 이 파울리 항들로 **순방향·역방향 노이즈 바운드**를 계산하고, 이것이 음영 라이트콘을 정의합니다.
- 나중에 노이즈 학습이 이 고정된 파울리 기저에 **층별 파울리-린드블라드 계수**를 채워 넣습니다.

순서를 정리하면 이렇습니다. SLC는 먼저 회로 구조와 관측량의 국소성으로부터 노이즈가 어디서 문제가 될 수 있는지를 정하고, 노이즈 학습은 나중에 그것이 얼마나 문제가 되는지를 정합니다.

In [ ]:
noise_model_paulis = generate_noise_model_paulis(unique_layer_slc, backend.coupling_map, boxed_circuit_slc)
noise_model_rates = {ref: None for ref in noise_model_paulis}

### 3.2.5 순방향·역방향 바운드 계산하기

회로의 어느 부분이 관측량에 영향을 줄 수 있는지 알아내기 위해 순방향 바운드와 역방향 바운드를 계산합니다. 설정을 조정하면 라이트콘의 모양과 색이 달라져서, 회로의 각 영역이 최종 측정에 어떻게 영향을 주는지 볼 수 있습니다. 이 분석은 노이즈 학습 전에 시뮬레이션으로 돌아가므로 QPU 시간을 쓰지 않습니다.

바운드는 `compute_forward_bounds`와 `compute_backward_bounds` 함수로 계산합니다. 두 함수 모두 박스 회로와 노이즈 모델 파울리 집합을 받습니다. `compute_forward_bounds`는 여기에 더해 ISA 레이아웃에 매핑된 관측량도 받습니다.

SLC 바운드 계산을 위한 추가 파라미터도 몇 개 정의합니다:
- `slc_atol` - 전개 과정에서 작은 파울리 항을 잘라 낼 절대 허용 오차. 계수가 이 문턱보다 작은 항은 계산 복잡도 관리를 위해 버립니다.
- `slc_eigval_max_qubits` - 교환자(commutator)의 고윳값 계산을 시도할 최대 큐비트 수. 이 값을 넘으면 바운드를 더 단순하고 느슨한 삼각 부등식으로 근사합니다.
- `slc_evolution_max_terms` - 시간 전개 동안 유지할 파울리 항의 최대 개수. 전개된 연산자의 크기를 제한해서 복잡도의 지수적 증가를 막습니다.
- `slc_num_processes` - 사용할 병렬 프로세스 수. 병렬 계산으로 순방향/역방향 바운드 계산을 빠르게 합니다.
- `slc_timeout` - 바운드 계산 하나에 허용하는 최대 시간(초). 개별 계산이 무한정 돌지 않게 막습니다.

이 파라미터 선택에 대한 자세한 내용은 [`compute_forward_bounds`](https://qiskit.github.io/qiskit-addon-slc/apidocs/qiskit_addon_slc.bounds.html#qiskit_addon_slc.bounds.compute_forward_bounds)와 [`compute_backward_bounds`](https://qiskit.github.io/qiskit-addon-slc/apidocs/qiskit_addon_slc.bounds.html#qiskit_addon_slc.bounds.compute_backward_bounds) 함수의 문서를 참고하세요.

In [ ]:
slc_atol = 1e-8
slc_eigval_max_qubits = 18
slc_evolution_max_terms = 1000
slc_num_processes = 8
slc_timeout = 300

*참고*: 다음 셀은 개인 컴퓨터에서 돌리면 45초 정도 걸립니다. `qBraid` 같은 클라우드 서비스를 쓰고 있다면 5분까지 걸릴 수 있습니다. 필요하면 위의 `slc_timeout`을 더 길게 조정하세요.

In [ ]:
forward_bounds = compute_forward_bounds(
    boxed_circuit_slc,
    noise_model_paulis,
    observable_local_isa,
    evolution_max_terms=slc_evolution_max_terms,
    eigval_max_qubits=slc_eigval_max_qubits,
    atol=slc_atol,
    num_processes=slc_num_processes,
    timeout=slc_timeout,
)

backward_bounds = compute_backward_bounds(
    boxed_circuit_slc,
    noise_model_paulis,
    evolution_max_terms=slc_evolution_max_terms,
    num_processes=slc_num_processes,
    timeout=slc_timeout,
)

이제 순방향·역방향 라이트콘을 플롯할 수 있습니다:

In [ ]:
print("forward bounds:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_slc,
            forward_bounds,
            noise_model_paulis,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )

print("backward bounds:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_slc,
            backward_bounds,
            noise_model_paulis,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            wire_order=wire_order_slc,
            measure_arrows=False,
        )
    )

노란 영역은 노이즈에 대한 민감도가 높은 곳입니다(여기서 생긴 오류는 관측량에 더 큰 영향을 줍니다). 어두운 영역은 민감도가 낮은 곳입니다(여기의 오류는 덜 중요합니다). 중요한 점은, 아직 노이즈를 학습하지 않았으므로 이 그림들이 실제 노이즈가 어디에 있는지를 보여 주는 것이 아니라는 사실입니다. 이 그림들이 보여 주는 것은, 만약 어떤 위치에 노이즈가 있다면 그것이 관심 관측량에 얼마나 강하게 영향을 줄 것인가입니다.

다음 절에서는 이 예제를 15큐비트로 키웁니다. 하드웨어에서 노이즈를 학습하고 음영 라이트콘과 결합해서, 낮은 샘플링 오버헤드로 오류를 완화할 것입니다.

### Exercise 5 — 15큐비트에서 3개 관측량의 국소성 조사하기

10큐비트, 트로터 10스텝의 이징 해밀토니안과 관측량 $Z_4$에 대해 음영 라이트콘(순방향·역방향 바운드)을 살펴봤습니다. 이 연습 문제에서는 예제를 15큐비트로 키우는 대신 트로터 스텝 수를 3으로 줄입니다.

SLC가 PEC보다 나은 점은 샘플링 오버헤드가 줄어든다는 것이고, 이는 관측량이 *국소적*일 때 특히 효과적입니다. 오버헤드가 얼마나 줄어드는지는 관측량의 국소성과 복잡도에 달려 있습니다. 이 연습 문제에서는 국소성이 서로 다른 세 관측량을 만들고 각각의 순방향·역방향 바운드를 계산합니다.

15큐비트, 트로터 3스텝의 미러 이징 회로를 설정하고 세 가지 관측량을 만듭니다:
- `obs_Z7` - 큐비트 7의 Z
- `obs_X3_Z11` - 큐비트 3의 X, 큐비트 11의 Z
- `obs_7_Zs` - 가운데 7개 큐비트의 Z

이 세 관측량 각각에 대해 순방향·역방향 바운드를 계산해야 합니다.

<div class="alert alert-block alert-success">

<b>Exercise 5.</b> 15큐비트에서 3개 관측량의 국소성 조사하기

- **트로터 3스텝짜리 15큐비트 이징 사슬 미러 회로를 만드세요.**
    - 함수 `construct_ising_circuit`을 사용하세요
- **필요한 3개 관측량을 `SparsePauliOp` 객체로 만드세요**:
    - `obs_Z7` - 큐비트 7의 Z
    - `obs_X3_Z11` - 큐비트 3의 X, 큐비트 11의 Z
    - `obs_7_Zs` - 이징 사슬 가운데 7개 큐비트의 Z들
    - 각 관측량은 계수가 1인 단일 항 관측량이어야 합니다.
- **음영 라이트콘 바운드를 계산하세요:**
    - 3개 관측량 각각에 대해:
        - 관측량을 물리(ISA) 큐비트 레이아웃에 매핑하세요.
        - `compute_forward_bounds`로 순방향 바운드를 계산하세요
        - 순방향 바운드를 `forward_bounds_list`에 추가하세요
    - 역방향 바운드는 관측량에 의존하지 않으므로 한 번만 계산하면 됩니다:
        - `compute_backward_bounds`로 역방향 바운드를 계산하세요


아래 셀을 채우세요.



</div>


*참고:* 순방향·역방향 바운드 계산에는 30초 정도 걸리니 조금만 기다려 주세요!

In [ ]:
# ========== Exercise ==========

# ============================================================
# TODO: Create 15-qubit Ising circuit
# By filling in the needed code below
# ============================================================

# Define the number of qubits and the number of Trotter steps.
num_qubits_ex5        = 15
num_trotter_steps_ex5 = 3
rx_angle         = np.pi / 8

# Build the mirrored Ising circuit.
circuit_ising_ex5 = ...
mirrored_circuit_ex5 = ...
mirrored_circuit_ex5.measure_all()

# Transpile to the backend ISA and capture the layout.
isa_circuit_ex5 = ...
final_layout_ex5 = isa_circuit_ex5.layout.final_index_layout()

# Box for SLC (uses `individual_modification` so we can scale per-error-term later).
boxed_circuit_ex5 = slc_boxing_pm.run(isa_circuit_ex5)
unique_layers_ex5 = find_unique_box_instructions(
    boxed_circuit_ex5, normalize_annotations=None, undress_boxes=True
)

# Generate noise model Pauli terms.
noise_model_paulis_ex5 = generate_noise_model_paulis(unique_layers_ex5, backend.coupling_map, boxed_circuit_ex5)
noise_model_rates_ex5 = {ref: None for ref in noise_model_paulis_ex5}

# ============================================================
# Construct required observables as SparsePauliOps
# ============================================================

# Z on qubit 7  →  one term: ("Z", [7], 1.0)
obs_Z7 = ...

# X on qubit 3, Z on qubit 11  →  one term: ("XZ", [3, 11], 1.0)
obs_X3_Z11 = ...

# Z on each of the 7 central qubits (one weight-7 Pauli product on qubits 4..10)
obs_7_Zs = ...

obs_list = [obs_Z7, obs_X3_Z11, obs_7_Zs]

# ============================================================
# Compute forward for each obs (loop over obs_list).
# ============================================================

slc_atol = 1e-8
slc_eigval_max_qubits = 18
slc_evolution_max_terms = 1000
slc_num_processes = 8
slc_timeout = 600

forward_bounds_list = []

for obs in obs_list:
    # your code goes here for calculating the forward bounds for all 3 observables

    # append your forward bounds to the list
    forward_bounds_list.append(forward_bounds)

# ========================================================================================
# Compute backward bounds - the backwards bounds does not depend on the observable in question, 
# so you just need to calculate it once
# ========================================================================================

backward_bounds = ...

In [ ]:
# grade your answer
grade_lab3_ex5(circuit_ising_ex5, 
    mirrored_circuit_ex5, 
    boxed_circuit_ex5, 
    obs_list, 
    forward_bounds_list, 
    backward_bounds
)

In [ ]:
check_progress("lab3")

아래 플롯들은 세 관측량의 순방향·역방향 바운드를 보여 줍니다.

In [ ]:
print(f"forward bounds for observable obs_Z7 :")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            forward_bounds_list[0],
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )


In [ ]:
print("forward bounds for observable obs_X3_Z11 :")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            forward_bounds_list[1],
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )


In [ ]:
print("forward bounds for observable obs_7_Zs :")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            forward_bounds_list[2],
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )


In [ ]:
print("backward bounds:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            backward_bounds,
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            wire_order=wire_order_slc,
            measure_arrows=False,
        )
    )

### 3.2.6 병합 바운드

이제 순방향 바운드와 역방향 바운드를 병합해서, 샘플링 오버헤드를 최소화하는 음영 라이트콘을 정의할 차례입니다.

#### 바운드는 왜 병합할까?

`merge_bounds` 함수는 회로 안에서 역방향 바운드가 순방향 바운드로 넘어가는 전환점을 고릅니다. 전환점은 총 추정 편향이 최소가 되도록 골라지고, 그 결과 조밀한 라이트콘이 만들어집니다.

여기에는 하드웨어에서 학습된 노이즈 레이트가 필요합니다. 회로의 어느 부분이 오류를 지배하는지 병합 알고리즘에게 알려 주는 것이 노이즈 모델이기 때문입니다.

#### 더 큰 회로에서 노이즈 다시 학습하기

Exercise 5에서 15큐비트로 키우면서 회로가 달라졌고, 고유 층들도 이전 10큐비트 PNA 설정에서 학습해 둔 노이즈 모델과는 다른 물리 큐비트 위에 놓이게 됐습니다. 그래서 바운드를 병합하기 전에 15큐비트 층들의 노이즈를 다시 학습해야 합니다. 노이즈 학습기 파라미터는 PNA에서 정의한 `pna_learner`를 재사용합니다.

*참고:* 다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `NOISE_LEARN_JOB_ID_SLC` 파라미터에 붙여넣고 `SUBMIT_NOISE_JOB_SLC = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 13초입니다(ibm_pittsburgh 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
NOISE_LEARN_JOB_ID_SLC = None   # paste job_id here on re-run
SUBMIT_NOISE_JOB_SLC   = False  # set True to submit a fresh learning job
pna_learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_SLC is not None:
    learner_job_slc = service.job(NOISE_LEARN_JOB_ID_SLC)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_SLC}")
elif SUBMIT_NOISE_JOB_SLC:
    learner_job_slc = pna_learner.run(unique_layers_ex5)
    NOISE_LEARN_JOB_ID_SLC = learner_job_slc.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_SLC}")
else:
    print("Set SUBMIT_NOISE_JOB_SLC=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_SLC and re-run.")


In [ ]:
learner_job_slc = service.job(NOISE_LEARN_JOB_ID_SLC)
print(f"{NOISE_LEARN_JOB_ID_SLC}  (status: {learner_job_slc.status()})")

In [ ]:
if learner_job_slc.status() == "DONE":
    noise_learner_result_slc = learner_job_slc.result()
else:
    print(f"Not done yet (status={learner_job_slc.status()}). Re-run cell when DONE.")

In [ ]:
if 'noise_learner_result_slc' in dir() and noise_learner_result_slc is not None:
    refs_to_noise_models_slc = noise_learner_result_slc.to_dict(unique_layers_ex5, require_refs=False)
    print(f"refs_to_noise_models_slc has {len(refs_to_noise_models_slc)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


#### 병합 바운드 계산하기

각 관측량에 대해, 학습된 노이즈 모델 `refs_to_noise_models_slc`와 함께 병합 바운드를 계산합니다.

*참고:* "*Optimal spacetime partitioning not implemented!Just partitioning list of noisy boxes.*"라는 경고 메시지가 보일 수 있습니다. 예상된 동작이며 오류가 아닙니다.

In [ ]:
# Merged bounds with the learned noise model — uses real Pauli-Lindblad rates,
# which typically shrinks the relevant lightcone substantially.
merged_bounds_with_noise = []
for i in range(3):
    merged_bounds_with_noise.append(merge_bounds(
        boxed_circuit_ex5,
        forward_bounds_list[i],
        backward_bounds,
        refs_to_noise_models_slc,
    ))


In [ ]:
print("Merged bounds with noise model for observable obs_Z7:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            merged_bounds_with_noise[0],
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )



### 3.2.7 샘플링 오버헤드 비교하기

이 절에서는 Exercise 5에서 만든 3개 관측량에 대해, SLC를 적용할 때와 PEC만 쓸 때의 샘플링 오버헤드를 계산합니다.

3.2.1절에서 PEC의 샘플링 오버헤드를 $\gamma^2$로 정의했습니다. 여기서 $$\gamma = \exp\!\big(2\sum_{l,\sigma}\lambda_{l,\sigma}\big),$$ 이고 $\lambda_{l,\sigma}$는 회로의 층 $l$에서 생성원 $\sigma$의 학습된 레이트입니다.

아래 셀에서는 $\gamma$를 위 식으로 직접 계산하는 것과 도우미 함수 `gamma_from_noisy_boxes`로 계산하는 것 두 가지로 구해서, 둘이 일치하는지 확인합니다.

In [ ]:
id_map = map_modifier_ref_to_ref(boxed_circuit_ex5)

# Manual γ from the Pauli-Lindblad formula: γ = exp(2 · Σ λ).
summed_rates = 0.0
for box_id, noise_id in id_map.items():
    learned_plm_ex5 = refs_to_noise_models_slc[noise_id]
    summed_rates += np.sum(learned_plm_ex5.rates)
total_gamma = np.exp(2 * summed_rates)

# Cross-check against the helper. This guards against edge cases
# (negative fitted rates, mis-mapped refs, etc.).
gamma_full_helper = gamma_from_noisy_boxes(refs_to_noise_models_slc, id_map)
assert np.isclose(total_gamma, gamma_full_helper), (
    f"Manual γ ({total_gamma:.6e}) disagrees with gamma_from_noisy_boxes ({gamma_full_helper:.6e})"
)

print(f"Full PEC γ = {total_gamma:.6f},  sampling cost γ² = {total_gamma**2:.6f}")
print(f"(helper agrees: γ = {gamma_full_helper:.6f})")


#### 국소 스케일 계산하기

`compute_local_scales` 함수는 회로에서 가능한 노이즈 생성원 하나하나를, 그 오류가 최종 측정의 편향에 얼마나 기여할 수 있는지로 순위를 매깁니다. 그 오류를 바로잡는 데 드는 완화 비용도 함께 고려합니다. 그런 다음 정해진 `bias_tolerance` 예산 안에서 편향을 최대한 줄여 주는 노이즈 생성원의 부분집합을 선택하여 다음의 세가지를 확인합니다:

- `local_scales`: 어떤 파울리를 적극적으로 완화하고 어떤 것은 내버려 둘지 말해 주는 생성원별 스케일 인자
- `sampling_cost`: 예측된 $\gamma^2$ 오버헤드
- `residual_bias_bound`: 일부 생성원을 완화하지 않고 남겨 둔 대가로 유지되는 편향

In [ ]:
# Bias-cost tradeoff sweep for all 3 observables (obs_Z7, obs_X3_Z11, obs_7_Zs).
# bias_tolerance from 0.001 to 0.101 in steps of 0.01 (no 0.0 — see note).
biases       = [[], [], []]
costs        = [[], [], []]
local_scales = [[], [], []]

bias_tolerance_values = np.arange(0.001, 0.102, 0.01).tolist()

for i in range(3):
    for bias in bias_tolerance_values:
        try:
            local_scale_, cost_, bias_ = compute_local_scales(
                boxed_circuit_ex5,                # circuit with noise boxes
                merged_bounds_with_noise[i],      # merged bounds for observable i
                refs_to_noise_models_slc,         # learned noise model rates
                sampling_cost_budget=np.inf,      # no cost budget — find optimal tradeoff
                bias_tolerance=bias,              # maximum allowed bias
            )
            biases[i].append(bias_)
            costs[i].append(cost_)
            local_scales[i].append(local_scale_)
        except (IndexError, ValueError):
            # compute_local_scales may not converge for every bound/bias combination
            # (depends on bounds structure); skip silently so the rest of the curve
            # still renders.
            continue


In [ ]:
xticks = np.arange(0, 11)

fig, ax = plt.subplots(figsize=(8, 5))

# Full-PEC reference (no SLC truncation).
ax.axhline(y=total_gamma ** 2, color="tab:orange", linestyle="--", linewidth=2, label="Full PEC")

# Bias-cost tradeoff curves for all 3 observables.
observable_names = ["Z on q7", "X on q3, Z on q11", "Z on 7 central qubits"]
colors           = ["tab:blue", "tab:green", "tab:red"]

for i in range(3):
    if not biases[i]:
        continue  # sweep produced no points for this observable
    ax.plot(
        100 * np.array(biases[i]),
        np.array(costs[i]),
        "o-",
        c=colors[i],
        label=f"SLC ({observable_names[i]})",
    )

ax.set_yscale("log")
ax.set_xticks(xticks, [f"{x:.1f}" for x in xticks])
ax.set_xlabel("Remaining bias [%]", fontsize=12)
ax.set_ylabel(r"Sampling overhead, $\gamma^2$", fontsize=12)
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right")

fig.suptitle("PEC sampling-overhead reduction enabled by SLC", fontsize=13)
plt.tight_layout()
plt.show()


플롯은 세 관측량 각각에 대해 샘플링 오버헤드 $\gamma^2$ 대 잔여 편향을 보여 주고, 전체 PEC 값을 수평 기준선으로 표시합니다. 몇 가지 짚어 보면:

- 큐비트 7의 국소 단일 큐비트 관측량(파란색)의 오버헤드가 가장 작습니다
- 2-체 관측량인 큐비트 3의 X와 큐비트 11의 Z(초록색)는 순방향·역방향 라이트콘이 더 많은 층을 덮기 때문에 더 높은 곳에 있습니다
- 큐비트 4–10에 Z가 걸린 7큐비트 관측량(빨간색)은 라이트콘이 가장 넓습니다.
- 전반적인 편향 허용치 범위 전체에서 세 오버헤드 모두 전체 PEC보다 낮게 유지됩니다.

다음 절에서는 가장 국소적인 관측량 `obs_Z7`에 집중해서, QPU에서 회로를 실행하며 전체 PEC와 SLC를 비교합니다.

### 3.2.8 오류 완화 작업을 QPU에서 실행하기

관측량 하나, 국소 단일 큐비트 관측량 `obs_Z7`에 대해 QPU 작업을 실행합니다.

In [ ]:
# Operating point: bias_tolerance = 0.02 (recommended trade-off between
# sampling cost reduction and residual bias bound). Observable: obs_Z7.
chosen_bias_thres = 0.02

local_scales_chosen, sampling_cost_chosen, residual_bias_chosen = compute_local_scales(
    boxed_circuit_ex5,
    merged_bounds_with_noise[0],   # obs_Z7 → index 0
    refs_to_noise_models_slc,
    sampling_cost_budget=np.inf,
    bias_tolerance=chosen_bias_thres,
)
print(f"Operating point: bias_tolerance = {chosen_bias_thres}")
print(f"  sampling cost (γ²)  = {sampling_cost_chosen:.3f}  "
      f"(vs full PEC γ² = {total_gamma**2:.3f})")
print(f"  residual bias bound = {100*residual_bias_chosen:.1f}%")

박스 회로의 `ChangeBasis` 어노테이션에서 기저 참조를 꺼내고, `samplex_inputs_slc`를 통해 측정 기저를 바인딩합니다.

- 기저 인코딩: $0=I$, $1=Z$, $2=X$, $3=Y$.

(`get_annotation`과 `ChangeBasis`는 노트북 맨 위에서 임포트했습니다)

In [ ]:
# Build template circuit + samplex for the boxed_circuit_ex5 (15-qubit).
template_circuit_slc, samplex_slc = samplomatic.build(boxed_circuit_ex5)

# Find the measurement box and extract its ChangeBasis ref.
meas_boxes_slc = [
    inst for inst in boxed_circuit_ex5.data
    if inst.operation.name == "box"
    and get_annotation(inst.operation, ChangeBasis) is not None
]


basis_ref_slc = get_annotation(meas_boxes_slc[0].operation, ChangeBasis).ref


# Canonical → physical → virtual mapping for boxed_circuit_ex5.
canonical_qubits_ex5 = [
    i for i, q in enumerate(boxed_circuit_ex5.qubits)
    if q in meas_boxes_slc[0].qubits
]
p_2_v_ex5 = {p: v for v, p in enumerate(final_layout_ex5)}
c_2_v_ex5 = {c: p_2_v_ex5[p] for c, p in enumerate(canonical_qubits_ex5)}

# Bases for obs_Z7 (single Pauli term → single basis) in canonical order.
bases_virt_obs1, reverser_virt_obs1 = get_measurement_bases(obs_Z7)
meas_bases_slc_canonical = [
    np.array([base[c_2_v_ex5[c]] for c in range(15)], dtype=np.uint8)
    for base in bases_virt_obs1
]

In [ ]:
# Build samplex_inputs for TWO experiments in the same Executor job:
#   - "unmitigated" : noise_scales = 0  (no anti-noise injected)
#   - "PEC+SLC"     : noise_scales = -1 + local_scales (anti-noise, pruned)

# Unmitigated: noise_scales = 0 on every per-instance ref.
samplex_inputs_unmit  = {f"noise_scales.{ref}": 0.0 for ref in local_scales_chosen}
samplex_inputs_unmit |= {"basis_changes": {basis_ref_slc: meas_bases_slc_canonical[0]}}
samplex_args_unmit = samplex_slc.inputs().make_broadcastable().bind(**samplex_inputs_unmit)

# PEC+SLC: noise_scales = -1 + local_scales for SLC pruning.
samplex_inputs_slc  = {f"noise_scales.{ref}": -1.0 for ref in local_scales_chosen}
samplex_inputs_slc |= {"basis_changes": {basis_ref_slc: meas_bases_slc_canonical[0]}}
samplex_inputs_slc |= {"local_scales": local_scales_chosen}
samplex_args_slc = samplex_slc.inputs().make_broadcastable().bind(**samplex_inputs_slc)

In [ ]:
# Parameters for the SLC executor job — tuned for tight stats with minimal QPU.
# Each samplex_item gets these many shots; we have 2 items (unmit + PEC+SLC).
shots_per_randomization_slc = 64
num_randomizations_slc      = 256

# Build the QuantumProgram. noise_maps at program level — keyed by unique-layer
# refs from the SLC NoiseLearner result.
program_slc = QuantumProgram(
    shots=shots_per_randomization_slc,
    noise_maps=refs_to_noise_models_slc,
)

# Item 0: unmitigated baseline (noise_scales = 0)
program_slc.append_samplex_item(
    circuit=template_circuit_slc,
    samplex=samplex_slc,
    samplex_arguments=samplex_args_unmit,
    shape=(num_randomizations_slc,),
)
# Item 1: PEC+SLC (noise_scales = -1, with local_scales)
program_slc.append_samplex_item(
    circuit=template_circuit_slc,
    samplex=samplex_slc,
    samplex_arguments=samplex_args_slc,
    shape=(num_randomizations_slc,),
)

print(f"Program built: 2 samplex_items (unmit + PEC+SLC), "
      f"{num_randomizations_slc} randomizations × {shots_per_randomization_slc} shots each")

*참고:* 다음 셀은 실제 양자 하드웨어에서 작업을 실행합니다. 실행할 준비가 되었는지 확인한 뒤 진행하세요. 처음 한 번 실행한 뒤에는 작업 ID를 `EXECUTOR_JOB_ID_SLC` 파라미터에 붙여넣고 `SUBMIT_EXECUTOR_JOB_SLC = False`로 바꿔 두기를 권장합니다. 작업을 실수로 다시 제출하는 일을 막아 주고, 커널을 재시작해도 노트북을 처음부터 다시 돌릴 수 있게 해 줍니다.

*예상 QPU 실행 시간은 12초입니다(ibm_fez 기준).* 이 추정치는 백엔드 실행 시간만 반영한 것으로, 대기열, 캘리브레이션, 런타임 세션 지연은 이보다 길 수 있습니다.

In [ ]:
#  qpu time = 12s, tested on ibm_fez

executor = Executor(backend)

EXECUTOR_JOB_ID_SLC      = None    # paste job_id here after first submission
SUBMIT_EXECUTOR_JOB_SLC  = False     # set True to submit a fresh job
executor.options.environment.job_tags = ["qgss26"]

if EXECUTOR_JOB_ID_SLC is not None:
    job_exec_slc = service.job(EXECUTOR_JOB_ID_SLC)
    print(f"Re-using saved job: {EXECUTOR_JOB_ID_SLC}")
elif SUBMIT_EXECUTOR_JOB_SLC:
    job_exec_slc = executor.run(program_slc)
    EXECUTOR_JOB_ID_SLC = job_exec_slc.job_id()
    print(f"Submitted: {EXECUTOR_JOB_ID_SLC}")
else:
    print("Set SUBMIT_EXECUTOR_JOB_SLC=True to submit a fresh job, "
          "or paste a saved job id into EXECUTOR_JOB_ID_SLC and re-run.")

In [ ]:
if EXECUTOR_JOB_ID_SLC is not None:
    if job_exec_slc.status() == "DONE":
        exec_results_slc = job_exec_slc.result()
        print(f"Got {len(exec_results_slc)} samplex_item results "
              f"(item 0 = unmitigated, item 1 = PEC+SLC).")
    else:
        print(f"Not done yet: {job_exec_slc.status()}")

### 3.2.9 결과 분석

In [ ]:
# Extract two results from the same Executor job:
#   results[0] → unmitigated (noise_scales=0, no gamma_factor)
#   results[1] → PEC+SLC      (noise_scales=-1, local_scales applied)

# γ for PEC+SLC normalization
gamma_slc = gamma_from_noisy_boxes(refs_to_noise_models_slc, id_map, local_scales_chosen)

# Item 0 — Unmitigated
res_unmit_slc = executor_expectation_values(
    exec_results_slc[0]["meas"],
    reverser_virt_obs1,
    meas_basis_axis=None, avg_axis=0,
    measurement_flips=exec_results_slc[0]["measurement_flips.meas"],
    pauli_signs=exec_results_slc[0].get("pauli_signs", None),
    rescale_factors=None,
    gamma_factor=1.0,
)
exp_val_unmit_slc = res_unmit_slc[0][0]
std_unmit_slc     = res_unmit_slc[0][1]

# Item 1 — PEC+SLC
res_slc = executor_expectation_values(
    exec_results_slc[1]["meas"],
    reverser_virt_obs1,
    meas_basis_axis=None, avg_axis=0,
    measurement_flips=exec_results_slc[1]["measurement_flips.meas"],
    pauli_signs=exec_results_slc[1].get("pauli_signs", None),
    rescale_factors=None,
    gamma_factor=gamma_slc,
)
exp_val_unmit_slc, var_unmit_slc = res_unmit_slc[0]
std_unmit_slc = np.sqrt(var_unmit_slc)

ideal_obs_Z7 = 1.0   # Z@7 on a Trotter mirror with |0⟩ initial state → +1

print(f"{'Method':<14} {'<O_Z7>':>10} {'std':>12} {'γ²':>10}")
print("─" * 50)
print(f"{'Unmitigated':<14} {exp_val_unmit_slc:>+10.4f} {std_unmit_slc:>12.4e} {'1.000':>10}")
print(f"{'PEC+SLC':<14} {exp_val_slc:>+10.4f} {std_slc:>12.4e} {gamma_slc**2:>10.3f}")
print(f"{'Ideal':<14} {ideal_obs_Z7:>+10.4f}")
print()
print(f"Sampling-overhead reduction: {total_gamma**2 / gamma_slc**2:.2f}× vs full PEC  "
      f"(γ² {total_gamma**2:.2f} → {gamma_slc**2:.2f})")
print(f"Mitigation: {(exp_val_slc - exp_val_unmit_slc) / (ideal_obs_Z7 - exp_val_unmit_slc) * 100:.1f}% "
      f"of the unmitigated error closed")

# Comparison plot
fig, ax = plt.subplots(figsize=(7, 4.5))
methods = ["Unmitigated", "PEC+SLC"]
values  = [exp_val_unmit_slc, exp_val_slc]
errs    = [std_unmit_slc,     std_slc]
colors  = ["tab:gray", "tab:purple"]

x = np.arange(len(methods))
ax.errorbar(x, values, yerr=errs, fmt="o", markersize=14, capsize=6,
            color="black", ecolor="gray", linewidth=2, zorder=3)
for xi, vi, ci in zip(x, values, colors):
    ax.scatter([xi], [vi], color=ci, s=120, zorder=4)

ax.axhline(ideal_obs_Z7, color="green", linestyle="--", linewidth=2,
           label=f"Ideal = {ideal_obs_Z7}", zorder=2)

ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel(r"$\langle Z_7 \rangle$", fontsize=13)
ax.set_title(f"15-qubit Ising mirror — PEC+SLC at bias_tolerance={chosen_bias_thres}",
             fontsize=12)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
#ax.set_ylim(0, max(1.2, max(values) * 1.2))
plt.tight_layout()
plt.show()

### 3.2.10 결론

SLC 기법의 핵심 장점은 샘플링 비용 절감입니다. 목표 관측량의 전파된 라이트콘 안에 있는 노이즈 생성원만 남기면, 선택한 `bias_tolerance`에서 전체 PEC 오버헤드 $\gamma^2$가 떨어집니다. 편향과 비용의 트레이드오프입니다. 작은 편향을 허용하는 대가로 샘플링 오버헤드를 크게 줄이는 것입니다.

이 15큐비트 이징 미러에서 $Z_7$ 같은 국소 관측량이라면 라이트콘이 충분히 좁아서 가지치기를 해도 기댓값이 보존됩니다. 복원된 $\langle Z_7\rangle$은 이상값 1에 가깝고, 샘플링 오버헤드는 순수 PEC보다 줄어듭니다.

이것으로 3장을 마칩니다. 여기에서는 구조를 아는 오류 완화 전략 세 가지를 다뤘으며 이 기법들은 모두 2장의 Samplomatic + NoiseLearnerV3 토대 위에 세워져 있습니다.

- PNA: 관측량을 다시 써서 반노이즈를 흡수
- PEC: 회로를 반노이즈로 다시 쓰기
- SLC: 음영 라이트콘으로 PEC의 노이즈 생성원 가지치기

Lab 3을 완주한 것을 축하합니다!

다음 Lab 4에서는 양자 우위라는 개념과 그것을 측정하는 방법을 소개합니다. 실습에는 IBM의 Quantum Advantage Tracker를 활용하는 것도 포함됩니다. 또한 오류 완화 기법과 큐비트 수를 줄이는 영리한 인코딩 전략을 갖춘 양자 최적화 알고리즘을 구현하면서, 이 아이디어들을 실제로 시연해 보게 됩니다.

# 추가 정보

**제작:** Sophy Shin & Sophie Engineer